# M6 · Pen-mark detector for marked workshop estimates

A surveyor reviews a workshop's repair estimate with a pen: a cross or a line through a row they do not accept, and a new figure next to an amount they want changed. **M6 finds those pen marks on the page and proposes which printed row each one belongs to.** This notebook trains the detector behind it and takes it to the point where the application can load it.

What the notebook does, in order:

1. Loads the **corrected page images that the real M4 page-reading module produced** for the synthetic dataset `data/raw/synthetic_motor_claims_500_w_annotation`, and the dataset's mark labels.
2. Looks at the data: mark styles, distractors, box shapes, coordinate frames.
3. Fine-tunes a **Faster R-CNN** object detector (ResNet-50 FPN, pretrained on COCO) to find two classes: `exclusion` and `price_change`.
4. Selects score thresholds on the validation split and measures detection.
5. Runs the project's existing row linker on the detections and measures detection plus linking per row: with rows built from ground truth the way the specification defines a row box, and with the rows a trained M5 model produced when they are available.
6. Exports a registry **candidate** and loads it back with the same loader the application uses.

### What you will learn

- What an object detector consumes and returns, and why its input is a *list* of images.
- How Faster R-CNN works: backbone, feature pyramid, anchors, region proposals, box head, non-maximum suppression.
- Why box *shape* matters (a pen strike is up to 190 times wider than it is tall) and how to check anchors against your data before training.
- How to read loss curves, precision-recall curves, average precision, and an error analysis.
- Why a detection score threshold is a decision you make on validation data, and how to make it.
- Why the same linker gives very different results depending on the row boxes it is handed.
- Why "the model reloads without an error" is not proof that it is the same model, and what does count as evidence.

### Where M6 sits

```mermaid
flowchart LR
    U["Uploaded estimate<br/>(photo or PDF)"] --> M4["M4 page reading<br/>corrected page + OCR text"]
    M4 -->|"text boxes"| M5["M5 line items<br/>rows + amounts"]
    M4 -->|"corrected page image"| D["M6 detector<br/>(this notebook)"]
    D -->|"mark boxes + scores"| L["M6 linker<br/>(existing, rule-based)"]
    M5 -->|"row boxes"| L
    L -->|"pending marks"| M8["M8 checks"]
    L --> M9["M9 review:<br/>surveyor confirms"]
    M9 -->|"confirmed marks"| M8
```

The same picture is drawn with matplotlib further down, in case your viewer does not render Mermaid.

> **Status, stated plainly**
>
> - The dataset is **synthetic**: one page layout, and "handwriting" rendered from three fonts. Every number this notebook prints is *synthetic validation evidence*. None of it is performance on real photographed, hand-marked estimates.
> - M6 **proposes, it never decides.** Every mark it produces is `pending` until a surveyor confirms, rejects or relinks it. A `price_change` box says *where* a handwritten amount is. It is not a recognised number.
> - The row linker is used exactly as it is in the repository. Its thresholds are proposed and untuned.
> - The linker needs row boxes from M5. The main linking results use **oracle rows**: ground-truth rows built as the M6 specification defines a row box, the union of the row's printed cells. Section 8.7 also measures the chain M4 → M5 → M6 with the rows of **one trained M5 candidate** when its rows file is present, and shows as a contrast what happens when rows are drawn as whole table bands.
> - Validation pages choose the training epoch, the score thresholds **and** the reported numbers. The operating point is therefore optimistic; section 8.2 says by how much it could move.
> - This is stage 1 of the training specification (synthetic pages only). Stage 2, fine-tuning on real team-marked pages, is out of scope here: those pages do not exist yet.
> - The exported model is a **candidate**. Nothing here installs it into the application.

## 1. Environment and paths

Use the project's Python environment as the Jupyter kernel (**CLAIM-CMEV (Python 3.12)**). The first cell finds the repository root and points the model caches at `artifacts/models/.cache/`, which must happen *before* `torch` is imported. The pretrained detector weights are already in that cache, so no download is needed.

**Tip for long runs on a laptop.** Training takes tens of minutes. macOS may put an idle machine to sleep in the middle of it. Keep the machine awake while the notebook runs (for example by starting Jupyter with `caffeinate -i jupyter lab`). If a run is interrupted anyway, section 7 shows how to resume it.

In [ ]:
from pathlib import Path
import os
import sys

# Works when Jupyter starts in the repository root or a subdirectory such as notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "docs/specs/README.md").is_file() and (p / "pipelines").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the CLAIM-CMEV checkout, or set ROOT to its path.")
for entry in (str(ROOT), str(ROOT / "src")):
    if entry not in sys.path:
        sys.path.insert(0, entry)
# Set caches before importing torch; preserve any user overrides.
os.environ.setdefault("HF_HOME", str(ROOT / "artifacts/models/.cache/huggingface"))
os.environ.setdefault("TORCH_HOME", str(ROOT / "artifacts/models/.cache/torch"))
print("Repository:", ROOT)
print("Kernel Python:", sys.executable)

The next cell is **disabled**. Set `INSTALL_DEPENDENCIES = True` only if the kernel is missing packages; it installs into the kernel's own environment. Restart the kernel afterwards.

In [ ]:
INSTALL_DEPENDENCIES = False  # Set True only to install into this kernel's environment.
if INSTALL_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(ROOT / "requirements-training.txt"), "-e", str(ROOT) + "[vision]"])
    print("Restart the kernel after installation, then rerun this notebook.")

Imports and versions. Three helper modules do the reusable work and have unit tests under `tests/unit/m6/`:

| Module | What it holds |
| --- | --- |
| `pipelines/documents/penmarks_data.py` (`pmd`) | Loading labelled pages on M4's corrected renders, augmentation, the torch dataset |
| `pipelines/documents/penmarks_training.py` (`pmt`) | Building the network, the training loop, reloading a run, export |
| `pipelines/documents/penmarks_evaluation.py` (`pme`) | Matching, average precision, threshold selection, linking evaluation |

The serving side is `src/claim_cmev/documents/pen_marks/detector.py` (`serving`). It builds the network for both training and serving, so the two cannot drift apart.

In [ ]:
from collections import Counter
from dataclasses import asdict, replace
from datetime import datetime, timezone
import inspect
import json
import math
import re
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import yaml
from IPython.display import Markdown, display
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle
from PIL import Image

from claim_cmev.contracts.common import COST_BASIS
from claim_cmev.contracts.documents import PageTransform, effective_price_for
from claim_cmev.documents.pen_marks import (
    MarkAction, apply_mark_action, link_detections, load_linking_config, load_penmark_detector, score_row,
)
from claim_cmev.documents.pen_marks import detector as serving
from claim_cmev.vision.registry import ModelUnavailable
from pipelines.documents import penmarks_data as pmd
from pipelines.documents import penmarks_evaluation as pme
from pipelines.documents import penmarks_training as pmt
from pipelines.documents.m4_cache import (
    M4CacheError, corrected_boxes_to_norm, load_m4_cache, source_to_corrected_boxes,
)
from pipelines.vision.training import select_device

print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__, "| NumPy:", np.__version__)
print("CUDA available:", torch.cuda.is_available(), "| Apple MPS available:", torch.backends.mps.is_available())

# One colour-blind-safe palette (Okabe-Ito) for every figure.
PALETTE = pmd.PALETTE
CLASS_NAMES = {"exclusion": "exclusion", "price_change": "price change"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 130, "font.size": 10, "axes.titlesize": 11,
                     "axes.labelsize": 10, "legend.fontsize": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.constrained_layout.use": True})
pd.set_option("display.max_colwidth", 80, "display.width", 160)

## 2. Configuration: the one cell to edit

Everything you might change is in the next cell.

| Switch | Default | Meaning |
| --- | --- | --- |
| `QUICK_RUN` | `False` | `True` trains on 96 pages for three epochs, to check that every cell runs. It takes a few minutes. **Its numbers are not a model result**, and the notebook says so in its output. |
| `FINAL_EVAL` | `False` | `True` opens the held-out **test** split, once, after the model and thresholds are frozen (section 11). |
| `EXPORT_FOR_SERVING` | `False` | `True` writes a registry candidate under `REGISTRY_ROOT` (section 12). |
| `DEVICE` | `None` | `None` picks CUDA, then Apple MPS, then CPU. Or name one: `"cuda"`, `"mps"`, `"cpu"`. |
| `RELOAD_RUN_DIR` | `None` | A saved run folder: skip training and evaluate that run (section 9). |
| `RELOAD_OVERWRITES_RUN_OUTPUTS` | `False` | With `False`, a reload writes its figures and reports to `artifacts/evaluation/<run_id>/reload-<time>/` and leaves the run's own figures and `validation_report.json` as they are. `True` replaces them. |
| `RESUME_RUN_DIR` | `None` | A saved run folder: continue an interrupted run from its last finished epoch. |
| `M5_ROWS_FILE` | `"auto"` | Rows from a trained M5 model for the linking evaluation (section 8.7). `"auto"` takes the newest `artifacts/evaluation/lineitems-*/rows_for_m6_validation.json` if there is one; a path names a file; `None` skips that part. |
| `EXPORT_NAME` | `None` | Name of the registry entry. `None` means `<run_id>-serving`. An entry is never overwritten, so exporting a saved run again needs a new name. |

The recipe values are explained next to the training run in section 7. **Every value is proposed**: none has been accepted by the team, and all were chosen on synthetic data.

In [ ]:
# ---- switches ---------------------------------------------------------------------------
QUICK_RUN = False            # True: 96 pages, three epochs. Checks plumbing only; NOT a model result.
FINAL_EVAL = False           # True only once, after model and thresholds are frozen (section 11).
EXPORT_FOR_SERVING = False   # True writes a registry candidate (section 12). Never overwrites.
DEVICE = None                # None = auto: CUDA, then Apple MPS, then CPU. Or "cuda", "mps", "cpu".
RELOAD_RUN_DIR = None        # e.g. ROOT / "artifacts/models/penmarks/<run_id>": evaluate a saved run.
RELOAD_OVERWRITES_RUN_OUTPUTS = False   # False: a reload writes to <run_id>/reload-<time>/ and keeps the run's own outputs.
RESUME_RUN_DIR = None        # e.g. the same folder: continue an interrupted run.
M5_ROWS_FILE = "auto"        # rows from a trained M5 model for section 8.7: "auto", a path, or None to skip.
EXPORT_NAME = None           # None = "<run_id>-serving". Exporting a saved run again needs a new name.

# ---- inputs and outputs -------------------------------------------------------------------
DATASET_DIR = ROOT / "data/raw/synthetic_motor_claims_500_w_annotation"
M4_CACHE_DIR = ROOT / "data/processed/synthetic_motor_claims_500_w_annotation/m4-v1"
M4_CACHE_RECEIPT = ROOT / "data/manifests/synthetic_motor_claims_m4_cache_20261012.json"
ARTIFACTS_ROOT = ROOT / "artifacts"          # runs: artifacts/models/penmarks/<run_id>, figures: artifacts/evaluation/<run_id>
REGISTRY_ROOT = ROOT / "artifacts/models"    # export: <REGISTRY_ROOT>/penmarks/<EXPORT_NAME>
M5_ROWS_ROOT = ROOT / "artifacts/evaluation" # where M5_ROWS_FILE = "auto" looks for lineitems-*/rows_for_m6_validation.json
CREATED_BY = "Lane 3 (documents), notebooks/M6_Document_Worker.ipynb"

# ---- detector (all PROPOSED) --------------------------------------------------------------
ANCHORS = "torchvision_default"  # or "pen_marks_wide"; section 4.6 measures both
MIN_SIZE, MAX_SIZE = 800, 1333   # working resolution: shorter side, cap on the longer side; see section 4.5

# ---- training recipe (all PROPOSED; section 7 explains each value) -----------------------
RECIPE = dict(
    batch_size=4, accumulation_steps=1,      # or batch_size=2, accumulation_steps=2 on a small GPU
    epochs=20, learning_rate=0.005, momentum=0.9, weight_decay=5e-4,
    warmup_steps=100, lr_milestones=(0.7, 0.9), lr_gamma=0.1,
    patience=5, seed=20260922, trainable_backbone_layers=3,
    score_floor=0.05, nms_iou=0.5, max_detections_per_page=50,
)
AUGMENT = pmd.AugmentConfig()    # training-only; set to None to train without augmentation. Section 4.7 previews it.
OVERRIDES = {}                   # e.g. {"epochs": 12}; keep this visible when you report a run
QUICK = dict(train_pages=96, val_pages=24, epochs=3, warmup_steps=8)   # used only when QUICK_RUN is True

# ---- derived: do not edit below -----------------------------------------------------------
settings = {**RECIPE, **pmt.ANCHOR_PRESETS[ANCHORS], "min_size": MIN_SIZE, "max_size": MAX_SIZE,
            "augment": AUGMENT, "device": DEVICE or "auto", "artifacts_root": str(ARTIFACTS_ROOT)}
settings.update(OVERRIDES)
if QUICK_RUN:
    settings.update(epochs=QUICK["epochs"], warmup_steps=QUICK["warmup_steps"])
if RELOAD_RUN_DIR is not None and RESUME_RUN_DIR is not None:
    raise ValueError("Set RELOAD_RUN_DIR or RESUME_RUN_DIR, not both.")
if RELOAD_RUN_DIR is not None:
    RUN_DIR = Path(RELOAD_RUN_DIR).expanduser().resolve()
    RUN_ID = RUN_DIR.name
    # A saved run is evaluated with the configuration it was trained with, not with the values above.
    cfg = pmt.TrainingConfig.from_record(json.loads((RUN_DIR / "config.json").read_text()))
else:
    RUN_ID = Path(RESUME_RUN_DIR).name if RESUME_RUN_DIR is not None else pmt.new_run_id() + ("-quick" if QUICK_RUN else "")
    cfg = pmt.TrainingConfig(run_id=RUN_ID, **settings)
    RUN_DIR = ARTIFACTS_ROOT / "models" / "penmarks" / RUN_ID
# RUN_EVAL_DIR belongs to the run (its final report lives there). EVAL_DIR is where THIS execution writes.
RUN_EVAL_DIR, EVAL_DIR = pmt.evaluation_dirs(ARTIFACTS_ROOT, RUN_ID, reload=RELOAD_RUN_DIR is not None,
                                             overwrite_run_outputs=RELOAD_OVERWRITES_RUN_OUTPUTS)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_NAME = EXPORT_NAME or f"{RUN_ID}-serving"
print("Run ID:", RUN_ID)
print("Checkpoints and history:", RUN_DIR)
print("Figures and reports:   ", EVAL_DIR)
if EVAL_DIR != RUN_EVAL_DIR:
    print("  (a reload: the run's own figures and reports in", RUN_EVAL_DIR.name + "/ are left as they are)")
elif RELOAD_RUN_DIR is not None:
    print("  (RELOAD_OVERWRITES_RUN_OUTPUTS is True: this reload REPLACES the run's own figures and validation report)")
if QUICK_RUN:
    print(f"\nQUICK_RUN is on: a few pages and {QUICK['epochs']} epochs unless a run is resumed or reloaded. "
          "Every number below checks plumbing only. It is NOT a model result.")

**Device and seeds.** The detector runs in 32-bit floats without mixed precision: on Apple's MPS backend mixed precision fails inside the region-pooling operator, and it is not needed here.

The run is seeded, so the data order and the augmentations repeat exactly. On a GPU some operations are still not bit-reproducible. **A repeat run will differ slightly** (expect differences in the third decimal of the validation scores). That is normal; it is one reason a difference of one or two validation marks between two runs should not be read as a real improvement.

`save_show` is the small helper every figure goes through: it saves the figure under `artifacts/evaluation/<run_id>/` and then shows it.

In [ ]:
device = select_device(cfg.device)
print("Selected device:", device)
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed % 2**32)

_figure_number = 0


def save_show(fig, name):
    """Save a figure under EVAL_DIR as NN_name.png, then show it."""
    global _figure_number
    _figure_number += 1
    path = EVAL_DIR / f"{_figure_number:02d}_{name}.png"
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.show()


def note(text):
    """Print a QUICK_RUN reminder where a number could be mistaken for a result."""
    if QUICK_RUN:
        print("QUICK_RUN: plumbing check only, not a model result.", text)


def shown(path):
    """A path relative to the repository when it lies inside it."""
    path = Path(path)
    return path.relative_to(ROOT) if path.is_relative_to(ROOT) else path

## 3. Inputs: what M6 receives

### 3.1 The M4 cache

M6 does not read the uploaded file. It reads what **M4 (page reading)** produced from it: a *corrected page image* (deskewed or perspective-corrected when M4 could do that reliably) and a *page transform* that maps between the uploaded file and that corrected image. Training on anything else would teach the detector a geometry it will never see when it is served.

Running M4 means running OCR, which is slow (about four seconds per page). So M4 was run once over all 500 pages of the dataset, and its real outputs were stored in a **cache**: per page the `DocumentPage` record, the corrected image `rectified.png` and the page-reading artifact. The loader verifies every file against its recorded SHA-256, and checks that the dataset on disk is the one the cache was built from.

If the cache is missing, the cell stops with the exact command that builds it.

In [ ]:
try:
    cache = load_m4_cache(M4_CACHE_DIR, DATASET_DIR, verify="hashes", splits=("train", "validation"))
except (M4CacheError, FileNotFoundError) as exc:
    receipt = json.loads(M4_CACHE_RECEIPT.read_text()) if M4_CACHE_RECEIPT.is_file() else {}
    command = receipt.get("build_command", "(build command not found: see docs/m4-ocr-worker.md)")
    raise RuntimeError(
        f"The M4 cache at {M4_CACHE_DIR} is missing or does not verify:\n  {exc}\n\n"
        "Build it by running the real M4 adapter over the dataset. This needs Docker and the OCR image "
        "(infra/Dockerfile.ocr) and takes about 35 minutes for 500 pages:\n\n"
        f"  {command}\n\n"
        "Then verify it:\n"
        f"  python -m pipelines.documents.m4_cache build --dataset {DATASET_DIR} --cache {M4_CACHE_DIR} --verify-only"
    ) from exc

counts = cache.manifest["counts"]
print(f"M4 cache verified: {counts['pages']} pages, schema {cache.manifest['schema']}")
print("  pages per split:   ", counts["by_split"])
print("  M4 page status:    ", counts["by_page_status"])
print("  geometry correction:", counts["by_geometry_correction"])
print("  M4 versions:        ", {k: cache.manifest["m4"]["versions"][k] for k in ("m4_code", "page_config", "ocr_weights")})

### 3.2 The `DocumentPage` record and the coordinate frames

Each cached page carries a validated `DocumentPage`, the same record the application stores. The parts M6 cares about:

| Field | Meaning |
| --- | --- |
| `page_id` | M4's identifier of the page. Marks and rows refer to the page by it. |
| `corrected_render_ref` | Reference (with SHA-256) to the corrected page image. `rectified.png` in the cache is exactly that file. |
| `transform` | `PageTransform`: the source size, the corrected size, the kind of correction, and a 3 × 3 **homography** matrix that maps source pixels to corrected pixels (plus its inverse). |
| `quality.state` | `complete`, `partial` or `unreadable`: how well M4 could *read* the page. |
| `text_boxes` | The OCR text. M5 uses it. The detector does not. |

> **Concept: coordinate frame.** A box is four numbers, and they only mean something together with the frame they are measured in. This notebook uses four frames: *dataset pixels* (the dataset's JPEG), *corrected-page pixels* (M4's output; the detector's input and output), the detector's *internal resized pixels*, and *normalised* 0-1 coordinates (corrected pixels divided by the page width and height), which is what the linker and the stored `PenMark` records use.

In [ ]:
example_cached = cache.by_id("SYN-00001_P2")
document_page = example_cached.page
transform = document_page.transform
print("image id:            ", example_cached.image_id, "| split:", example_cached.split, "| page type:", example_cached.page_type)
print("DocumentPage.page_id:", document_page.page_id)
print("corrected image:     ", shown(example_cached.rectified_path))
print("  sha256 matches the record:", document_page.corrected_render_ref.sha256 == example_cached.entry["rectified_sha256"])
print("source size:         ", transform.source_width, "x", transform.source_height, "px")
print("corrected size:      ", transform.corrected_width, "x", transform.corrected_height, "px")
print("geometry correction: ", transform.geometry_correction, f"({transform.correction_reason})")
print("homography (source -> corrected):")
print(np.array(transform.homography))
print("quality.state:       ", document_page.quality.state, "|", len(document_page.text_boxes), "OCR text boxes (used by M5, not here)")

identity = sum(np.allclose(p.page.transform.homography, np.eye(3)) for p in cache.pages if p.split != "test")
considered = sum(p.split != "test" for p in cache.pages)
print(f"\nIdentity transforms among train and validation pages: {identity} of {considered}")

**What to notice.** On this dataset every homography is the identity matrix and the correction is `none`: the synthetic pages are perfectly flat scans, so M4 found nothing to correct. Dataset pixels and corrected-page pixels are therefore the same numbers *here*.

Real photographs will not be this kind. A page photographed at an angle is warped by M4, and a box labelled on the photograph lands somewhere else on the corrected page. So this notebook maps every labelled box through the page transform anyway, with `source_to_corrected_boxes`. The next cell shows what that helper does when a transform is *not* the identity: a made-up transform that shrinks the page by 10 % and rotates it by 3 degrees.

In [ ]:
from types import SimpleNamespace

angle, shrink = math.radians(3.0), 0.9
made_up = np.array([[shrink * math.cos(angle), -shrink * math.sin(angle), 80.0],
                    [shrink * math.sin(angle), shrink * math.cos(angle), 20.0],
                    [0.0, 0.0, 1.0]])
warped_page = SimpleNamespace(transform=PageTransform(
    source_width=1240, source_height=1754, corrected_width=1240, corrected_height=1754, exif_orientation=1,
    geometry_correction="perspective", correction_reason="illustration_only",
    homography=made_up.tolist(), homography_inverse=np.linalg.inv(made_up).tolist()))
label_box = [831, 658, 1127, 704]   # a price-change mark, in dataset pixels
print("box in dataset pixels:          ", label_box)
print("identity transform (this data): ", source_to_corrected_boxes(document_page, label_box).round(1).tolist()[0])
print("made-up warp (a real photo):    ", source_to_corrected_boxes(warped_page, label_box).round(1).tolist()[0])
print("\nWith a warp the four corners move separately; the helper returns the upright box around them.")

### 3.3 The labelled pages

The dataset's detection labels are COCO files (`marks_coco_train.json`, `marks_coco_validation.json`): one box per *logical mark*, with category `exclusion` or `price_change`. A logical mark includes its handwritten remark: a row strike with "Not accepted" written next to it is **one** box around both.

`annotations.jsonl` holds more: the mark **style**, the **distractors** (ticks and review notes, which are *not* targets), and for every printed **row** its printed field boxes (quantity, description, action, amount, remarks) and the table band it sits in. `load_penmark_pages` reads both sources, refuses to continue if they disagree, and maps every box into the corrected-page frame.

A row gets one **row box**: the smallest box around its printed fields. That is how the M6 specification defines the box of a row (step 4: "the union of its cell boxes"), and it is the box M5 hands to M6. The taller table band is kept beside it; section 8.7 shows why the difference matters.

Three rules are enforced here and asserted in code:

- **The test split is not loaded.** It is the final held-out partition and is read only in section 11, behind `FINAL_EVAL`.
- **Splits are grouped by claim.** A claim has two pages (an assessment page and an estimate page). Both are in the same partition, so nothing about a claim leaks from training into validation.
- **A page M4 could not read is still a page.** Three estimate pages are `unreadable` for M4 (its OCR output was inconsistent). They still have a corrected image, so the detector trains and predicts on them. What they lack is M5 rows; section 8.7 comes back to that.

In [ ]:
pages = pmd.load_penmark_pages(cache, DATASET_DIR)          # train and validation only
pmd.assert_claim_grouped(cache.pages)                       # no claim in two partitions (all three splits)
train_pages_all = [p for p in pages if p.split == "train"]
val_pages_all = [p for p in pages if p.split == "validation"]
assert not {p.dataset_claim_id for p in train_pages_all} & {p.dataset_claim_id for p in val_pages_all}
assert not any(p.split == "test" for p in pages)


def first_claims(members, page_count):
    """The pages of the first claims, both pages of each claim together."""
    claims = list(dict.fromkeys(p.dataset_claim_id for p in members))[:max(1, page_count // 2)]
    return [p for p in members if p.dataset_claim_id in claims]


train_pages = first_claims(train_pages_all, QUICK["train_pages"]) if QUICK_RUN else train_pages_all
val_pages = first_claims(val_pages_all, QUICK["val_pages"]) if QUICK_RUN else val_pages_all
data_record = (json.loads((RUN_DIR / "run.json").read_text())["data"] if RELOAD_RUN_DIR is not None
               else pmd.data_record(cache, DATASET_DIR, pages))

display(pd.DataFrame(pmd.split_rows(pages)).set_index("split"))
print(f"Training on {len(train_pages)} pages, validating on {len(val_pages)} pages.")
unreadable = [p.image_id for p in pages if p.page_status == "unreadable"]
print("Pages M4 could not read (kept; they still have a corrected image):", unreadable)
note("Only a subset of the pages is used for training and validation.")

**How to read the table.** `pages_without_target` is large on purpose: every assessment page and about one estimate page in eight has no mark to find. A detector must also learn when to say nothing. `acceptance_tick` and `review_note` are counted but they are *distractors*, not targets.

### 3.4 Do the mapped boxes land on the ink?

A wrong transform would put every box in the wrong place, silently. So before anything is trained, the labels are checked against the pixels. The picture shows one corrected page with every box that was mapped onto it. The cell then checks **all** train and validation pages: every labelled box must contain ink, and the ink inside it must reach all four sides of the box to within two pixels.

In [ ]:
def page_pixels(page):
    return pmd.load_image(page)


def draw_page(axis, page, pixels=None, *, rows=True, distractors=True, crop=None, label=True):
    """A corrected page with its ground-truth boxes in the notebook's colours."""
    axis.imshow(page_pixels(page) if pixels is None else pixels)
    if rows:
        pme.draw_boxes(axis, [r.box for r in page.rows], PALETTE["row"], linewidth=0.8)
        pme.draw_boxes(axis, [r.amount_box for r in page.rows if r.amount_box], PALETTE["amount"], linewidth=1.0, pad=2)
    for mark_class in pmd.TARGET_CLASSES:
        marks = [m for m in page.marks if m.mark_class == mark_class]
        pme.draw_boxes(axis, [m.box for m in marks], PALETTE[mark_class], pad=3,
                       labels=[f"{CLASS_NAMES[mark_class]}: {m.style}" for m in marks] if label else None)
    if distractors:
        for mark_class in pmd.DISTRACTOR_CLASSES:
            pme.draw_boxes(axis, [m.box for m in page.distractors if m.mark_class == mark_class], PALETTE[mark_class],
                           pad=3, linestyle=":")
    if crop is not None:
        axis.set_xlim(crop[0], crop[2])
        axis.set_ylim(crop[3], crop[1])
    axis.set_xticks([])
    axis.set_yticks([])


def table_region(page, margin=40):
    """The part of an estimate page that holds its table rows (x0, y0, x1, y1), or the whole page."""
    if not page.rows:
        return (0, 0, page.width, page.height)
    boxes = np.array([r.band for r in page.rows])   # the table bands: the whole parts table
    return (max(0, boxes[:, 0].min() - margin), max(0, boxes[:, 1].min() - margin - 60),
            min(page.width, boxes[:, 2].max() + margin), min(page.height, boxes[:, 3].max() + margin))


example = pmd.records_by_id(pages)["SYN-00001_P2"]
fig, axes = plt.subplots(1, 2, figsize=(15, 8.5), gridspec_kw={"width_ratios": [1, 1.75]})
draw_page(axes[0], example, label=False)
axes[0].set_title(f"{example.image_id}: M4's corrected page ({example.width} x {example.height} px)")
draw_page(axes[1], example, crop=table_region(example))
axes[1].set_title("The parts table, enlarged")
handles = [Rectangle((0, 0), 1, 1, fill=False, edgecolor=PALETTE[k], linestyle=s, label=t) for k, s, t in [
    ("exclusion", "-", "target: exclusion"), ("price_change", "-", "target: price change"),
    ("acceptance_tick", ":", "distractor: tick"), ("review_note", ":", "distractor: review note"),
    ("row", "-", "row box: union of the row's printed cells"), ("amount", "-", "printed amount (ground truth)")]]
fig.legend(handles=handles, loc="outside lower center", ncol=6, frameon=False)
fig.suptitle("Ground-truth boxes mapped into the corrected-page frame", fontsize=13)
save_show(fig, "ground_truth_overlay")

# Check every labelled box of train and validation against the pixels.
without_ink, deviations = 0, []
for page in pages:
    if not (page.marks or page.distractors):
        continue
    mask = pmd.ink_mask(page_pixels(page))
    for mark in (*page.marks, *page.distractors):
        extent = pmd.ink_extent(mask, mark.box)
        if extent is None:
            without_ink += 1
        else:
            deviations.append(np.abs(np.array(extent) - np.array(mark.box)).max())
deviations = np.array(deviations)
print(f"Boxes checked: {len(deviations) + without_ink} (targets and distractors, train and validation)")
print(f"  boxes holding no ink: {without_ink}")
print(f"  largest gap between a box side and the ink inside it: {deviations.max():.0f} px")
assert without_ink == 0 and deviations.max() <= 2, "the mapped boxes do not sit on the ink"

**What to look for.** Coloured solid boxes are the two target classes. Dotted boxes are distractors. Grey boxes are the row boxes (for each row, the smallest box around its printed cells) and orange boxes the printed amounts; the detector never sees those, the *linker* does. Notice that a row box is only as wide as the row's printed text: a row with an empty remarks cell ends at its amount. Outlines are drawn a few pixels outside the true box so that thin marks stay visible.

Note that the ink check cannot use colour: the synthetic surveyor writes in several blues and in near-black, like real pens. The detector cannot rely on "blue means pen" either.

## 4. Understanding the data

Everything in this section uses the **training** split (and the validation split for counts). The test split is not touched.

### 4.1 What the targets look like

The dataset draws each class in several **styles**. The M6 specification gives an X over a row and a handwritten number as its examples; strike-only and remark-only exclusions go beyond those examples, so every result later in this notebook is reported per style as well as per class.

| Class | Style | What the surveyor drew |
| --- | --- | --- |
| `exclusion` | `x` | A small cross in the row |
| `exclusion` | `x_and_remark` | A cross plus a written remark such as "Not accepted" |
| `exclusion` | `row_strike` | A line through the whole row |
| `exclusion` | `row_strike_and_remark` | A line through the row plus a remark |
| `exclusion` | `remark` | Only a written remark |
| `price_change` | `amount_strike` | A line through the printed amount, and a new amount written in the remarks column |
| `price_change` | `amount_cross` | Crossing strokes over the printed amount, and a new amount written in the remarks column |

Each row of the picture shows three training examples of one style, cut out of the page with the labelled box drawn.

In [ ]:
def crop_around(box, page, min_width=260, min_height=96, pad=14):
    """A crop window around a box, at least min_width x min_height, inside the page."""
    cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
    half_w = max(box[2] - box[0], min_width) / 2 + pad
    half_h = max(box[3] - box[1], min_height) / 2 + pad
    return (max(0, cx - half_w), max(0, cy - half_h), min(page.width, cx + half_w), min(page.height, cy + half_h))


def mark_gallery(groups, title, name, examples=3, row_height=1.35):
    """One row per (class, style): `examples` crops from training pages with the labelled box drawn."""
    found = {key: [] for key in groups}
    for page in train_pages_all:
        for mark in (*page.marks, *page.distractors):
            key = (mark.mark_class, mark.style)
            if key in found and len(found[key]) < examples:
                found[key].append((page, mark))
    fig, axes = plt.subplots(len(groups), examples, figsize=(15, row_height * len(groups) + 0.6), squeeze=False)
    for row, key in enumerate(groups):
        for column in range(examples):
            axis = axes[row, column]
            axis.set_xticks([])
            axis.set_yticks([])
            if column >= len(found[key]):
                axis.axis("off")
                continue
            page, mark = found[key][column]
            window = crop_around(mark.box, page)
            axis.imshow(page_pixels(page))
            pme.draw_boxes(axis, [mark.box], PALETTE[mark.mark_class], pad=3, linewidth=1.8)
            axis.set_xlim(window[0], window[2])
            axis.set_ylim(window[3], window[1])
            if column == 0:
                axis.set_ylabel(f"{key[1]}", rotation=0, ha="right", va="center", fontsize=9)
            width, height = mark.box[2] - mark.box[0], mark.box[3] - mark.box[1]
            axis.set_title(f"{page.image_id}  ({width:.0f} x {height:.0f} px)", fontsize=8)
    fig.suptitle(title, fontsize=12)
    save_show(fig, name)


TARGET_STYLES = [("exclusion", "x"), ("exclusion", "x_and_remark"), ("exclusion", "row_strike"),
                 ("exclusion", "row_strike_and_remark"), ("exclusion", "remark"),
                 ("price_change", "amount_strike"), ("price_change", "amount_cross")]
mark_gallery(TARGET_STYLES, "Detector targets by style (training pages; box size in the titles)", "target_styles")

**What to look for.**

- The box covers the **whole logical mark**. For a strike with a remark that is the line *and* the handwriting, which makes a box about 880 pixels wide and 30 pixels tall.
- A **price change is two things far apart**: the struck-out printed amount and the replacement written in the remarks column. One box spans both.
- The `x` is tiny, about 20 × 23 pixels on a 1240 × 1754 page.
- A `remark`-only exclusion is just handwriting. To tell it from the handwriting of a price change the detector has to use what the writing looks like (words against digits) and what surrounds it.

### 4.2 What the detector must ignore

Not every pen mark is a decision. The dataset adds two kinds of **distractor**, and the M6 specification puts ticks and notes out of scope:

- an **acceptance tick** next to a row the surveyor accepts;
- a **review note** on the assessment page, such as "See marked estimate".

They are not labelled as targets. They matter because they look like targets: a tick is the size of an X, and a review note is handwriting like a remark.

In [ ]:
mark_gallery([("acceptance_tick", "tick"), ("review_note", "handwritten_note")],
             "NOT targets: the detector must learn to ignore these (ticks and review notes)", "distractors")

### 4.3 How many, and what shape

Four views of the labels. Read them with one question in mind: *what will be hard for a detector that was pretrained on photographs of everyday objects?*

In [ ]:
style_table = pd.DataFrame(pmd.style_rows(pages))
display(style_table.round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
# Marks per page.
most = max(len(p.marks) for p in pages)
positions = np.arange(most + 1)
for offset, (split, members, colour) in enumerate([("train", train_pages_all, PALETTE["price_change"]),
                                                   ("validation", val_pages_all, PALETTE["amount"])]):
    histogram = Counter(len(p.marks) for p in members)
    bars = axes[0].bar(positions + (offset - 0.5) * 0.4, [histogram.get(i, 0) for i in positions], width=0.4,
                       color=colour, label=f"{split} ({len(members)} pages)")
    axes[0].bar_label(bars, fontsize=7)
axes[0].set(xlabel="Target marks on the page", ylabel="Pages", xticks=positions,
            title="Marks per page: most pages have none")
axes[0].legend()
# Style counts.
targets = style_table[style_table["target"]]
labels = [f"{c}\n{s}" for c, s in zip(targets["mark_class"], targets["style"])]
y = np.arange(len(targets))
axes[1].barh(y - 0.2, targets["train"], height=0.4, color=PALETTE["price_change"], label="train")
axes[1].barh(y + 0.2, targets["validation"], height=0.4, color=PALETTE["amount"], label="validation")
for index, (a, b) in enumerate(zip(targets["train"], targets["validation"])):
    axes[1].text(a + 1, index - 0.2, str(a), va="center", fontsize=8)
    axes[1].text(b + 1, index + 0.2, str(b), va="center", fontsize=8)
axes[1].set(yticks=y, xlabel="Marks", title="Target marks per style")
axes[1].set_yticklabels(labels, fontsize=8)
axes[1].invert_yaxis()
axes[1].legend()
save_show(fig, "counts")

**How to read these.**

- *Left.* Half the pages are assessment pages with no target, and some estimate pages have none either. A page with no mark is a legitimate training example, not a mistake.
- *Right.* The two classes are balanced, but validation is small: **each style has between 6 and 24 validation marks**. One missed mark moves a style's recall by 4 to 17 percentage points. Per-style numbers later in this notebook are therefore counts first and percentages second.

In [ ]:
train_marks = [(p, m) for p in train_pages_all for m in p.marks]
widths = np.array([m.box[2] - m.box[0] for _, m in train_marks])
heights = np.array([m.box[3] - m.box[1] for _, m in train_marks])
styles = [m.style for _, m in train_marks]
STYLE_MARKERS = {"x": "X", "x_and_remark": "P", "row_strike": "_", "row_strike_and_remark": "s", "remark": "o",
                 "amount_strike": "^", "amount_cross": "v"}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
for (mark_class, style) in TARGET_STYLES:
    chosen = np.array([s == style for s in styles])
    axes[0].scatter(widths[chosen], heights[chosen], s=22, alpha=0.6, marker=STYLE_MARKERS[style],
                    color=PALETTE[mark_class], label=f"{CLASS_NAMES[mark_class]}: {style}")
tick_boxes = np.array([d.box for p in train_pages_all for d in p.distractors])
axes[0].scatter(tick_boxes[:, 2] - tick_boxes[:, 0], tick_boxes[:, 3] - tick_boxes[:, 1], s=6, alpha=0.25,
                color=PALETTE["row"], label="distractors")
span = np.array([10, 1300])
for ratio in (1, 4, 16, 64):
    axes[0].plot(span, span / ratio, color="black", linewidth=0.6, linestyle="--")
    label_x = min(1300, 300 * ratio)      # where the line leaves the plot: at the top or at the right
    axes[0].text(label_x, label_x / ratio, f" {ratio}:1", fontsize=8, va="center")
axes[0].set(xscale="log", yscale="log", xlim=(10, 2200), ylim=(3, 400), xlabel="Box width (px, corrected page)",
            ylabel="Box height (px)", title="Box shapes in training: wide and thin (dashed lines = width:height)")
axes[0].legend(fontsize=7, loc="upper left", ncol=2)
share = widths * heights / (example.width * example.height)
axes[1].hist(share * 100, bins=np.logspace(-2.2, 0.5, 40), color=PALETTE["price_change"])
axes[1].set(xscale="log", xlabel="Box area as a share of the page (%)", ylabel="Marks",
            title=f"Marks are small: median {np.median(share) * 100:.2f} % of the page")
save_show(fig, "box_shapes")

print(f"Training target boxes: {len(widths)}; width {widths.min():.0f} to {widths.max():.0f} px (median {np.median(widths):.0f}); "
      f"height {heights.min():.0f} to {heights.max():.0f} px (median {np.median(heights):.0f}).")
print(f"Width divided by height: median {np.median(widths / heights):.1f}, largest {(widths / heights).max():.0f}.")

**How to read these.**

- *Left*, on logarithmic axes. Objects in everyday photographs sit near the 1:1 line: roughly as wide as tall, give or take a factor of two. Only the `x` is there. Most marks are **4 to 60 times wider than tall**, and a bare row strike is close to 100:1, up to about 190:1. This motivates the anchor comparison in section 4.6; that short comparison cannot establish it as the main source of errors.
- *Right.* A mark covers a few hundredths of a percent to about one percent of the page. The detector is looking for small things on a large, mostly empty image.

### 4.4 Following one mark through the coordinate frames

A detector resizes its input internally, and it is easy to lose track of which frame a box is in. The next cell follows one price-change mark through every frame it passes.

1. **Dataset pixels**: where the label file puts it (`[x, y, width, height]` in COCO; converted to `x0, y0, x1, y1` here).
2. **Corrected-page pixels**: after `source_to_corrected_boxes`. The detector's input image is in this frame.
3. **Internal resized pixels**: torchvision scales the page so that its shorter side is `MIN_SIZE` (unless the longer side would exceed `MAX_SIZE`), then pads it to a multiple of 32. The network computes in this frame.
4. **Output boxes**: torchvision divides by the same scale before returning boxes, so the output is **already back in corrected-page pixels**. Nothing has to undo the resize a second time; doing so would be a bug.
5. **Normalised `box_norm`**: corrected pixels divided by the page width and height. The linker and the stored `PenMark` records use this frame.

In [ ]:
followed = next(m for m in example.marks if m.mark_class == "price_change")
coco = json.loads((DATASET_DIR / "marks_coco_train.json").read_text())
x, y, w, h = next(a["bbox"] for a in coco["annotations"] if a["mark_id"] == followed.mark_id)
dataset_box = np.array([x, y, x + w, y + h], dtype=float)
corrected_box = source_to_corrected_boxes(example_cached.page, dataset_box)[0]
scale, resized, padded = pmd.resized_size(example.height, example.width, min_size=cfg.min_size, max_size=cfg.max_size)
internal_box = corrected_box * scale
norm_box = corrected_boxes_to_norm(example_cached.page, corrected_box)[0]

frames = pd.DataFrame([
    {"frame": "1. dataset pixels (label file)", "x0": dataset_box[0], "y0": dataset_box[1], "x1": dataset_box[2], "y1": dataset_box[3],
     "frame size (w x h)": f"{example.width} x {example.height}"},
    {"frame": "2. corrected-page pixels (detector input)", **dict(zip(("x0", "y0", "x1", "y1"), corrected_box)),
     "frame size (w x h)": f"{example.width} x {example.height}"},
    {"frame": f"3. internal resized pixels (x {scale:.4f})", **dict(zip(("x0", "y0", "x1", "y1"), internal_box)),
     "frame size (w x h)": f"{resized[1]} x {resized[0]}, padded to {padded[1]} x {padded[0]}"},
    {"frame": "4. output box (divided by the scale again)", **dict(zip(("x0", "y0", "x1", "y1"), internal_box / scale)),
     "frame size (w x h)": f"{example.width} x {example.height}"},
    {"frame": "5. box_norm (corrected / page size)", **dict(zip(("x0", "y0", "x1", "y1"), norm_box)),
     "frame size (w x h)": "1 x 1"},
]).set_index("frame")
display(frames.round(4))
print(f"Mark {followed.mark_id}: {followed.style}, {corrected_box[2] - corrected_box[0]:.0f} x {corrected_box[3] - corrected_box[1]:.0f} px on the page, "
      f"{(internal_box[2] - internal_box[0]):.0f} x {(internal_box[3] - internal_box[1]):.0f} px inside the network.")

pixels = page_pixels(example)
small = np.asarray(Image.fromarray(pixels).resize((resized[1], resized[0]), Image.Resampling.BILINEAR))
canvas = np.zeros((padded[0], padded[1], 3), dtype=np.uint8)
canvas[:resized[0], :resized[1]] = small
fig, axes = plt.subplots(1, 3, figsize=(15, 6.2))
axes[0].imshow(pixels)
pme.draw_boxes(axes[0], [corrected_box], PALETTE["price_change"], pad=4, linewidth=2)
axes[0].set_title(f"Frames 1, 2 and 4: corrected page\n{example.width} x {example.height} px")
axes[1].imshow(canvas)
pme.draw_boxes(axes[1], [internal_box], PALETTE["price_change"], pad=3, linewidth=2)
axes[1].axhline(resized[0], color=PALETTE["highlight"], linewidth=1)
axes[1].set_title(f"Frame 3: what the network computes on\n{resized[1]} x {resized[0]} px, padded to {padded[1]} x {padded[0]} (black)")
axes[2].add_patch(Rectangle((0, 0), 1, 1, fill=False, edgecolor="black"))
axes[2].add_patch(Rectangle((norm_box[0], norm_box[1]), norm_box[2] - norm_box[0], norm_box[3] - norm_box[1],
                            fill=True, facecolor=PALETTE["price_change"], alpha=0.6))
axes[2].set(xlim=(-0.05, 1.05), ylim=(1.05, -0.05), aspect=example.height / example.width, xlabel="x / page width",
            ylabel="y / page height", title="Frame 5: box_norm, the frame of the linker\nand of stored PenMark records")
save_show(fig, "coordinate_frames")

**What to check.** Rows 1, 2 and 4 of the table hold the same numbers on this dataset: 1 equals 2 because M4's transform is the identity here, and 4 equals 2 because torchvision undoes its own resize. Row 3 is smaller by the scale factor. Row 5 is between 0 and 1.

### 4.5 Decision: working resolution

The training specification proposes the torchvision default: shorter side 800, longer side at most 1333. A 1240 × 1754 page then becomes about 800 × 1131 inside the network, a scale of 0.645. Does the thin ink survive that?

The next cell measures the marks in both frames and shows three thin marks at native resolution and after the resize, enlarged to the same display size.

In [ ]:
compare = []
for (mark_class, style) in TARGET_STYLES:
    h = np.array([m.box[3] - m.box[1] for _, m in train_marks if m.style == style])
    w = np.array([m.box[2] - m.box[0] for _, m in train_marks if m.style == style])
    compare.append({"class": mark_class, "style": style, "marks": len(h),
                    "median size, native (px)": f"{np.median(w):.0f} x {np.median(h):.0f}",
                    f"median size at {cfg.min_size}/{cfg.max_size} (px)": f"{np.median(w) * scale:.0f} x {np.median(h) * scale:.0f}",
                    "smallest height, native": h.min(), f"smallest height at {cfg.min_size}": round(h.min() * scale, 1)})
display(pd.DataFrame(compare))

thin = [next((p, m) for p, m in train_marks if m.style == "row_strike" and m.box[3] - m.box[1] <= 6),
        next((p, m) for p, m in train_marks if m.style == "x"),
        next((p, m) for p, m in train_marks if m.style == "remark")]
fig, axes = plt.subplots(2, 3, figsize=(15, 4.6))
for column, (page, mark) in enumerate(thin):
    pixels = page_pixels(page)
    window = crop_around(mark.box, page, min_width=200, min_height=60, pad=6)
    x0, y0, x1, y1 = (int(v) for v in window)
    if mark.style == "row_strike":   # the strike is 766 px wide: show its left end
        x1 = x0 + 260
    crop = pixels[y0:y1, x0:x1]
    reduced = np.asarray(Image.fromarray(pixels).resize((resized[1], resized[0]), Image.Resampling.BILINEAR))
    crop_small = reduced[int(y0 * scale):int(y1 * scale), int(x0 * scale):int(x1 * scale)]
    axes[0, column].imshow(crop, interpolation="nearest")
    axes[0, column].set_title(f"{mark.style}, native: box {mark.box[2] - mark.box[0]:.0f} x {mark.box[3] - mark.box[1]:.0f} px", fontsize=9)
    axes[1, column].imshow(crop_small, interpolation="nearest")
    axes[1, column].set_title(f"after the resize: box {(mark.box[2] - mark.box[0]) * scale:.0f} x {(mark.box[3] - mark.box[1]) * scale:.1f} px", fontsize=9)
    for axis in axes[:, column]:
        axis.set_xticks([])
        axis.set_yticks([])
fig.suptitle(f"Thin marks before and after the internal resize to {resized[1]} x {resized[0]} (each pixel drawn as a square)", fontsize=12)
save_show(fig, "resolution")

**What to look for.** The ink stays visible after the resize: strokes get fainter and blockier but do not vanish. What shrinks is the *box*: the thinnest row strike is 4 pixels tall on the page and about 2.6 pixels inside the network, and an X is about 13 × 15 pixels.

Whether that costs accuracy cannot be read off a picture. Section 4.6 reports what was measured, together with the anchor comparison.

### 4.6 Decision: anchors

> **Concept: anchor.** Faster R-CNN does not search for boxes from nothing. It lays a fixed grid of reference boxes, the **anchors**, over the image: at every position of every feature map, a few boxes of set sizes and shapes. For each anchor the network predicts two things: "is there an object here?" and "how should this anchor be moved and stretched to fit it?". An anchor counts as a positive training example for a mark when their **IoU** is at least 0.7.
>
> **Concept: IoU (intersection over union).** The area two boxes share, divided by the area they cover together. 1 means identical, 0 means no overlap.

torchvision's default anchors were designed for everyday objects: sizes 32, 64, 128, 256 and 512 pixels, each in three shapes (2:1, 1:1 and 1:2). The pen marks here are up to 190:1. If no anchor resembles a mark, the network has to learn a huge stretch from a poor starting point.

So before training, measure it: **for every training mark, what is the best IoU that any anchor reaches?** The next cell does this for the default anchors and for a wider set, `pen_marks_wide`: two sizes per pyramid level, from 16 to 512 pixels, in the shapes 32:1, 8:1, 2:1 and 1:1.

In [ ]:
anchor_models = {name: serving.build_model(cfg.detector_settings().model_copy(update=dict(preset)))
                 for name, preset in pmt.ANCHOR_PRESETS.items()}
anchor_grids = {name: pmt.rpn_anchors(model, height=example.height, width=example.width)
                for name, model in anchor_models.items()}
train_boxes = np.array([m.box for _, m in train_marks])
best_iou = {name: pmd.best_anchor_iou(train_boxes * grid["scale"], grid["anchors"]) for name, grid in anchor_grids.items()}
del anchor_models

anchor_table = []
for name, grid in anchor_grids.items():
    values = best_iou[name]
    anchor_table.append({"anchor set": name, "anchors per position": grid["anchors_per_location"],
                         "anchors on a page": len(grid["anchors"]), "median best IoU": np.median(values),
                         "marks with best IoU < 0.3": f"{(values < 0.3).sum()} of {len(values)}",
                         "marks with best IoU >= 0.5": f"{(values >= 0.5).sum()} of {len(values)}",
                         "marks with best IoU >= 0.7": f"{(values >= 0.7).sum()} of {len(values)}"})
display(pd.DataFrame(anchor_table).set_index("anchor set").round(3))

ANCHOR_COLOURS = {"torchvision_default": PALETTE["row"], "pen_marks_wide": PALETTE["price_change"]}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1, 1.5]})
for name, values in best_iou.items():
    axes[0].hist(values, bins=np.linspace(0, 1, 41), histtype="stepfilled", alpha=0.55, color=ANCHOR_COLOURS[name], label=name)
top = axes[0].get_ylim()[1]
for threshold, text in ((0.3, "below 0.3: background"), (0.7, "0.7 and above: positive")):
    axes[0].axvline(threshold, color="black", linestyle="--", linewidth=0.8)
    axes[0].text(threshold - 0.012, top * 0.98, text, fontsize=8, va="top", ha="right", rotation=90)
axes[0].set(xlabel="Best IoU any anchor reaches for a training mark", ylabel="Training marks",
            title=f"How well can the anchors start? ({len(train_boxes)} training marks)")
axes[0].legend(loc="upper left", bbox_to_anchor=(0.0, -0.18), ncol=2, frameon=False)
positions = np.arange(len(TARGET_STYLES))
for offset, (name, values) in enumerate(best_iou.items()):
    groups = [values[np.array([s == style for s in styles])] for _, style in TARGET_STYLES]
    boxes = axes[1].boxplot(groups, positions=positions + (offset - 0.5) * 0.36, widths=0.3, patch_artist=True,
                            showfliers=False, medianprops={"color": "black"})
    for patch in boxes["boxes"]:
        patch.set_facecolor(ANCHOR_COLOURS[name])
        patch.set_alpha(0.7)
axes[1].axhline(0.7, color="black", linestyle="--", linewidth=0.8)
axes[1].axhline(0.3, color="black", linestyle="--", linewidth=0.8)
axes[1].set(xticks=positions, ylim=(0, 1), ylabel="Best anchor IoU",
            title="The same, per style (box = middle half, line = median)")
axes[1].set_xticklabels([s for _, s in TARGET_STYLES], rotation=20, ha="right", fontsize=8)
axes[1].legend(handles=[Rectangle((0, 0), 1, 1, color=c, alpha=0.7, label=n) for n, c in ANCHOR_COLOURS.items()], loc="upper right")
save_show(fig, "anchor_iou")

**How to read this.** Further right is better. With the default anchors at this resolution **1 of the 507 training marks reaches 0.7** and 292 are below 0.3, the value under which an anchor is treated as background. Training still works to a degree, because torchvision always promotes the single best anchor of each mark to a positive ("low-quality matching"), but each mark then has one poor starting box instead of several good ones.

The wide set moves 422 of the 507 marks to 0.5 or above, and 231 to 0.7 or above. The bare **row strike stays the hardest**: at roughly 100:1 not even a 32:1 anchor fits it well.

On this evidence alone you would expect the wide set to train better. Keep that expectation in mind; the short test below does not settle it.

The next picture shows the two sets to scale against typical marks.

In [ ]:
def anchor_shapes(preset, level):
    """(width, height) of the anchors of one pyramid level, as torchvision builds them."""
    shapes = []
    for ratio in preset["anchor_aspect_ratios"]:
        for size in preset["anchor_sizes"][level]:
            shapes.append((size / math.sqrt(ratio), size * math.sqrt(ratio)))
    return shapes


typical = {style: (np.median(widths[np.array([s == style for s in styles])]) * scale,
                   np.median(heights[np.array([s == style for s in styles])]) * scale) for _, style in TARGET_STYLES}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharex=True, sharey=True)
for axis, (name, preset) in zip(axes, pmt.ANCHOR_PRESETS.items()):
    for level in range(4):
        for width, height in anchor_shapes(preset, level):
            axis.add_patch(Rectangle((-width / 2, -height / 2), width, height, fill=False, linewidth=0.9,
                                     edgecolor=ANCHOR_COLOURS[name], alpha=0.9))
    for (mark_class, style) in TARGET_STYLES:
        width, height = typical[style]
        axis.add_patch(Rectangle((-width / 2, -height / 2), width, height, fill=True, alpha=0.35,
                                 facecolor=PALETTE[mark_class], edgecolor=PALETTE[mark_class]))
    axis.set(xlim=(-330, 330), ylim=(140, -140), aspect="equal", xlabel="Pixels (internal resized frame)",
             title=f"{name}: {len(preset['anchor_sizes'][0]) * len(preset['anchor_aspect_ratios'])} anchors per position")
axes[0].set_ylabel("Pixels")
fig.suptitle("Anchor outlines of the four finest pyramid levels (lines) against the median mark of each style (filled)", fontsize=12)
save_show(fig, "anchor_shapes")

**What to look for.** Filled rectangles are median marks: orange-red for exclusions, blue for price changes. On the left, the default anchors are squares and mild rectangles; the long thin marks stick far out of every one of them. On the right, the wide set has long flat anchors that follow them, and a 16-pixel square small enough for the X.

**The trade-off.** The anchor count per position is the number of outputs of the region-proposal head. Changing it from 3 to 8 changes the shape of that head's two output layers, so their COCO-pretrained weights no longer fit and they **start from random values**. Everything else (backbone, feature pyramid, the shared 3 × 3 convolution of that head, the box head) keeps its pretrained weights. More anchors per position also means more anchors per page (about 614,000 instead of 230,000 at this resolution).

#### A short comparison, and why it is inconclusive

The IoU plot is a **prediction**: it says the wide anchors *should* make learning easier. A prediction has to be checked by training. One short run was made per anchor set: the recipe of section 7, **4 epochs** each, scored on the **validation** split (76 pages, 100 marks) at each run's best epoch and its own best thresholds. Counts are "marks found at IoU 0.5 / marks"; exclusion precision is "correct detections / detections kept".

| Anchors | Resolution | Selection score | Exclusion recall | Exclusion precision | Price-change recall | `x` | `x_and_remark` | `row_strike` | `row_strike_and_remark` | `remark` | AP@[.5:.95] excl. / price | Minutes per epoch |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `torchvision_default` | 800 / 1333 | 0.768 | 33/54 | 33/70 | 46/46 | 12/12 | 0/13 | 10/12 | 10/11 | 1/6 | 0.28 / 0.67 | 3.0 |
| `pen_marks_wide` | 800 / 1333 | 0.745 | 25/54 | 25/48 | 46/46 | 6/12 | 0/13 | 6/12 | 11/11 | 2/6 | 0.20 / 0.69 | 2.2 |

These two rows were recorded on 2026-10-12 with short runs outside this notebook (synthetic validation evidence, 4 epochs, **one run per row**; they are not recomputed here unless you switch on `RUN_DECISION_RUNS` in section 7). Minutes per epoch were measured while other jobs shared the GPU, so compare them only roughly.

**What the runs say about anchors: nothing conclusive.** After 4 epochs the default anchors were ahead by 0.023 on the selection score (0.768 against 0.745). That gap is smaller than what one run does from one epoch to the next: the full reference run (`penmarks-fasterrcnn-20261012T145839Z-8529778e`, 2026-10-12, default anchors) scored 0.767, 0.734 and 0.788 at its epochs 4, 5 and 6 with nothing changed but one more epoch of training. One run per arm cannot separate a difference that small from run-to-run noise.

The comparison is also not like for like at 4 epochs. The wide set changes the number of anchors per position, so the two output layers of its region-proposal head **start from random values** while the default arm keeps every pretrained weight. Four epochs measure mostly how far those new layers have caught up. They do not say which set is better once both are trained.

**Why the default anchors are kept all the same.** Not because they won this comparison. They are the training specification's default; they keep every pretrained RPN weight; and the full run in section 7 found 99 of the 100 validation marks with them (reference run above; section 8.4 prints the count for the run you load). `pen_marks_wide` stays available in the configuration cell. A real answer would need full-length runs of both sets and several seeds each; section 7 can repeat the arms.

**What was measured about resolution: almost nothing.** A run at native resolution (1240 / 1754, default anchors) was started on 2026-10-12 and **stopped after its first epoch** to free the GPU for the full run, so it is not in the table. After that one epoch its selection score was 0.483, against 0.488 for the same anchors at 800 / 1333 after one epoch. One epoch of one run is no evidence either way: it neither shows that native resolution helps nor that it does not. What is known is the cost. An earlier timing probe on this machine measured 0.84 s per batch of 2 pages at native resolution against 0.78 s per batch of 4 at 800 / 1333 (about 2.6 against 1.3 minutes per epoch on an idle GPU).

The default therefore stays at **800 / 1333** because it is the training specification's default and trains in half the time, not because a comparison decided it. Whether native resolution would find thin strikes more precisely is open. `RUN_DECISION_RUNS` in section 7 includes exactly that arm (default anchors at 1240 / 1754), so the comparison can be completed.

Both arms share one weakness after 4 epochs: `x_and_remark` and `remark` are rarely found. Those are the styles where the detector must read handwriting as "an exclusion remark" and, for `x_and_remark`, draw **one** box around a cross and a remark that can be far apart. They need more training than 4 epochs: the 20-epoch reference run finds all of them (section 13).

**A lesson worth keeping.** The anchor analysis was cheap and its conclusion sounded obvious. One short run per arm did not confirm it, and was too little to refute it. "Inconclusive" is a legitimate result. Say so, keep the documented default, and write down what a conclusive test would take.

### 4.7 Augmentation

> **Concept: augmentation.** During training each page is shown in a slightly altered version every epoch. The network cannot memorise 350 exact images, and it becomes less sensitive to things that should not matter, such as a darker scan.

For a detector, an augmentation is only correct if the **boxes stay correct**. So only two kinds are used:

- **Photometric** changes that do not move anything: brightness and contrast, a smooth lighting ramp across the page (a crude shadow), a mild blur, JPEG re-compression. Boxes are untouched.
- **Geometric** changes whose effect on a box can be computed exactly: a small zoom about the page centre and a small shift. An optional small rotation is available (`rotation_degrees`, off by default); each box then becomes the upright box around its rotated rectangle.

**No horizontal flip.** Flipping is standard for photographs, where a mirrored cat is still a cat. A page is not mirror-symmetric: the amount column is on the right, text runs left to right, and a tick mirrored is a different stroke. Flipped pages would teach the detector a layout that does not exist.

The preview shows the same table region six times: the original, then five training views with what was applied.

In [ ]:
if AUGMENT is None:
    print("AUGMENT is None: training uses the pages as they are.")
else:
    pixels = page_pixels(example)
    window = table_region(example)
    views = [("original", pixels, example.boxes, {})]
    for seed in range(4):
        out, boxes, labels, info = pmd.augment_page(pixels, example.boxes, example.labels, np.random.default_rng([cfg.seed, 0, seed]), AUGMENT)
        views.append((f"training view {seed + 1}", out, boxes, info))
    rotated = replace(AUGMENT, rotation_degrees=2.0)
    out, boxes, labels, info = pmd.augment_page(pixels, example.boxes, example.labels, np.random.default_rng(3), rotated)
    views.append(("optional: with rotation_degrees=2", out, boxes, info))
    fig, axes = plt.subplots(2, 3, figsize=(15, 6.6))
    for axis, (title, image, boxes, info) in zip(axes.ravel(), views):
        axis.imshow(image)
        pme.draw_boxes(axis, boxes, PALETTE["exclusion"], pad=3, linewidth=1.4)
        axis.set_xlim(window[0], window[2])
        axis.set_ylim(window[3], window[1])
        axis.set_xticks([])
        axis.set_yticks([])
        details = []
        if info:
            details.append(f"zoom {info['scale']:.3f}, shift ({info['translate_px'][0]:+.0f}, {info['translate_px'][1]:+.0f}) px"
                           + (f", rotate {info['rotation_degrees']:+.1f} deg" if info["rotation_degrees"] else ""))
            details.append(f"brightness {info['brightness']:.2f}, contrast {info['contrast']:.2f}"
                           + (f", blur {info['blur_radius']:.1f}" if info["blur_radius"] else "")
                           + (f", JPEG q{info['jpeg_quality']}" if info["jpeg_quality"] else "")
                           + (", shadow" if info["shadow"] else ""))
        axis.set_title(title + ("\n" + "\n".join(details) if details else ""), fontsize=8)
    fig.suptitle("Augmentation preview: the boxes move with the ink (table region of one training page)", fontsize=12)
    save_show(fig, "augmentation")
    display(pd.DataFrame([asdict(AUGMENT)]).T.rename(columns={0: "value (proposed)"}))

**What to look for.** In every view the boxes still sit on their marks. The changes are deliberately mild: this dataset has flat, clean pages, and validation pages are not augmented. Stronger photographic effects (perspective, heavy shadow) belong with real photographs, where M4's correction would be part of the picture.

## 5. The model: Faster R-CNN in plain terms

> **Concept: object detection.** Classification answers "what is in this image?". Detection answers "what is where?": a list of boxes, each with a class and a score between 0 and 1.

Faster R-CNN is a **two-stage** detector. Stage one proposes regions that might hold *something*. Stage two looks closely at each proposed region and decides *what* it is and where exactly its edges are.

```mermaid
flowchart LR
    I["Page image<br/>3 x H x W, values 0-1"] --> T["Transform<br/>normalise, resize, pad"]
    T --> B["Backbone: ResNet-50<br/>(pretrained)"]
    B --> F["Feature pyramid (FPN)<br/>5 maps, strides 4 to 64"]
    F --> R["Region proposal network (RPN)<br/>per anchor: objectness + box shift"]
    R -->|"about 1000 proposals"| P["RoI align<br/>crop 7 x 7 features per proposal"]
    F --> P
    P --> H["Box head<br/>2 fully connected layers"]
    H --> C["Box predictor<br/>3 class scores + box refinement"]
    C --> N["Per class: score floor,<br/>non-maximum suppression, cap"]
    N --> O["Boxes, labels, scores<br/>in input pixels"]
```

| Part | What it does |
| --- | --- |
| **Transform** | Normalises colours, resizes the page (section 4.4) and pads it. It also maps the final boxes back. It is *inside* the model, so the caller must not normalise or resize. |
| **Backbone (ResNet-50)** | A stack of convolutions that turns pixels into feature maps: grids of numbers that respond to strokes, corners, text texture. |
| **Feature pyramid (FPN)** | Combines backbone stages into five maps at strides 4, 8, 16, 32 and 64 pixels. Small marks are found on the fine maps, large ones on the coarse maps. |
| **Region proposal network (RPN)** | Slides over every feature map. For each anchor it outputs an *objectness* score and four numbers that shift and stretch the anchor. The best-scoring, non-overlapping results are the **proposals**. |
| **RoI align** | Cuts a fixed 7 × 7 grid of features out of the pyramid for each proposal, whatever the proposal's size and shape. |
| **Box head and predictor** | Two fully connected layers, then two outputs per proposal: scores for *background / exclusion / price change*, and a refined box for each class. |
| **Non-maximum suppression (NMS)** | Several proposals usually land on the same mark. Per class, the highest-scoring box is kept and boxes overlapping it by IoU 0.5 or more are dropped. |

The cell below draws the same pipeline with matplotlib.

In [ ]:
def block_diagram(blocks, title, name, width=15.5, height=2.6):
    """A left-to-right row of boxes joined by arrows. blocks: (heading, detail, colour)."""
    fig, axis = plt.subplots(figsize=(width, height))
    step = 1.0 / len(blocks)
    for index, (heading, detail, colour) in enumerate(blocks):
        left = index * step + 0.06 * step
        axis.add_patch(FancyBboxPatch((left, 0.16), 0.88 * step, 0.68, boxstyle="round,pad=0.006,rounding_size=0.02",
                                      facecolor=colour, edgecolor="black", linewidth=0.8, alpha=0.35))
        axis.text(left + 0.44 * step, 0.67, heading, ha="center", va="center", fontsize=9.5, fontweight="bold")
        axis.text(left + 0.44 * step, 0.40, detail, ha="center", va="center", fontsize=7.6)
        if index + 1 < len(blocks):
            axis.add_patch(FancyArrowPatch((left + 0.885 * step, 0.5), (left + 1.005 * step, 0.5), arrowstyle="-|>",
                                           mutation_scale=11, color="black", linewidth=0.9))
    axis.set(xlim=(0, 1), ylim=(0, 1))
    axis.axis("off")
    axis.set_title(title, fontsize=11)
    save_show(fig, name)


pretrained, new, fixed = PALETTE["price_change"], PALETTE["exclusion"], PALETTE["row"]
rpn_colour = new if ANCHORS != "torchvision_default" else pretrained
block_diagram([
    ("Page", "RGB, values 0-1\n3 x 1754 x 1240", fixed),
    ("Transform", f"normalise, resize to\n{resized[1]} x {resized[0]}, pad", fixed),
    ("ResNet-50", "backbone\npretrained (COCO)", pretrained),
    ("FPN", "5 feature maps\nstrides 4 to 64\npretrained", pretrained),
    ("RPN", "per anchor:\nobjectness + shift\n" + ("output layers NEW" if rpn_colour == new else "pretrained"), rpn_colour),
    ("RoI align", "7 x 7 features\nper proposal", fixed),
    ("Box head", "2 dense layers\npretrained", pretrained),
    ("Predictor", "3 class scores +\nbox refinement\nNEW", new),
    ("NMS", f"floor {cfg.score_floor}, IoU {cfg.nms_iou},\ncap {cfg.max_detections_per_page}", fixed),
    ("Output", "boxes, labels,\nscores", fixed),
], "Faster R-CNN ResNet-50 FPN as used here (blue: pretrained weights, orange-red: starts from random values, grey: no weights)",
    "model_diagram")

block_diagram([
    ("Uploaded\nestimate", "photo or PDF", fixed), ("M4", "page reading:\ncorrected page,\nOCR text", fixed),
    ("M5", "line items:\nrows, amounts\n(row boxes)", fixed), ("M6 detector", "THIS NOTEBOOK\nmark boxes + scores", new),
    ("M6 linker", "existing rules:\nmark -> row,\nall marks pending", pretrained), ("M9 review", "surveyor confirms,\nrejects, relinks,\ntypes the amount", fixed),
    ("M8 checks", "uses confirmed\nmarks only", fixed),
], "Where the detector sits (M4 also feeds M5; M5's rows feed the linker)", "pipeline_diagram", height=2.2)

### 5.1 What is pretrained and what is new

Training a detector from nothing needs far more than 350 pages. So the network starts from weights that torchvision's maintainers trained on **COCO**, a large set of everyday photographs with 80 object classes. Those weights already know how to turn pixels into useful features and how to propose regions. Fine-tuning adapts them to pen marks.

Two parts cannot be reused:

- The **class predictor** was built for 91 COCO category slots. It is replaced by one for 3 classes: background, `exclusion`, `price_change`. It starts from random values.
- Only with `ANCHORS = "pen_marks_wide"`: the **two output layers of the RPN** then have 8 outputs per position instead of 3, so they also start from random values (section 4.6). With the default anchors the whole RPN keeps its pretrained weights.

The first layers of the backbone stay **frozen** (`trainable_backbone_layers=3` trains only the last three of its five stages): edges and strokes look the same in photographs and on paper, and freezing them saves memory and guards against overfitting.

The network is built by `serving.build_model`, the function the application's loader uses, and then given the COCO weights. Building it the same way on both sides is deliberate; section 10 shows what goes wrong otherwise, and checks the result against a network that torchvision builds without that function.

In [ ]:
if RELOAD_RUN_DIR is not None:
    model_preview, _, preview_record = pmt.load_run(RUN_DIR, device="cpu")
    starting_point = preview_record["starting_point"]
else:
    model_preview, starting_point = pmt.build_detector(cfg)
parameters = pd.DataFrame(pmt.parameter_table(model_preview))
new_tensors = starting_point["new_tensors"]
parameters["starts from"] = ["COCO weights" if not any(t.startswith(b.split(" ")[0]) for t in new_tensors)
                             else "COCO weights, output layers random" if b.startswith("rpn") else "random values"
                             for b in parameters["block"]]
parameters.loc[len(parameters)] = ["total", parameters["parameters"].sum(), parameters["trainable"].sum(), ""]
display(parameters.style.format({"parameters": "{:,}", "trainable": "{:,}"}).hide(axis="index"))
print(f"Tensors copied from the COCO checkpoint: {starting_point['transferred_tensors']} of {len(model_preview.state_dict())}")
print("Tensors that start from random values:")
for name in new_tensors:
    print("  ", name, tuple(model_preview.state_dict()[name].shape))

base = starting_point["base_checkpoint"]
print("\nStarting checkpoint:", base["name"])
print("  file:   ", base["file"], "| sha256", base["sha256"])
print("  source: ", base["url"])
print("  COCO val2017 box mAP reported by torchvision:", base["coco_val2017_box_map"])
print("  licence:", base["licence"])

anchor_grid = pmt.rpn_anchors(model_preview, height=example.height, width=example.width)
levels = pd.DataFrame({"pyramid level": ["P2", "P3", "P4", "P5", "P6 (pooled)"],
                       "stride (px)": [4, 8, 16, 32, 64],
                       "feature map (h x w)": [f"{h} x {w}" for h, w in anchor_grid["feature_shapes"]],
                       "anchor sizes (px)": [", ".join(map(str, s)) for s in cfg.anchor_sizes],
                       "anchors": anchor_grid["anchors_per_level"]})
display(levels.style.format({"anchors": "{:,}"}).hide(axis="index"))
print(f"Anchors laid over one page: {len(anchor_grid['anchors']):,} "
      f"({anchor_grid['anchors_per_location']} per position; shapes h/w = {[round(r, 3) for r in cfg.anchor_aspect_ratios]})")

**How to read this.** Almost all of the 41 million parameters arrive pretrained. With the default anchors only the new predictor, 15,375 parameters, starts from random values. That is why a few hundred pages are enough to get a working detector.

**Licence note.** The code is torchvision's (BSD-3-Clause). The weights were trained by the torchvision maintainers on COCO train2017; their metadata states no licence of its own. The export records this in `NOTICE.md`.

The second table shows the feature pyramid for one page. Most anchors sit on the finest map (stride 4), which is where a 13-pixel X or a 3-pixel-tall strike has to be found.

## 6. One batch, step by step

Before a long training run, push one small batch through the network by hand. It shows what the data looks like to the model, and it catches shape and type mistakes in seconds.

A detection batch is **not one stacked tensor**. Images may differ in size and each has a different number of boxes, so the model takes:

- a **list of image tensors**, each `float32 [3, height, width]` with values 0-1 (RGB);
- a **list of target dictionaries**, one per image, with `boxes` as `float32 [N, 4]` in pixels of that image and `labels` as `int64 [N]` (1 = exclusion, 2 = price change; 0 is reserved for background and never appears in a target).

A page with no marks has `N = 0`: boxes of shape `[0, 4]`. The batch below pairs the example estimate page with an assessment page that has no marks.

In [ ]:
empty_page = next(p for p in train_pages_all if not p.marks)
walk = pmd.PenMarkDataset([example, empty_page])        # no augmentation
images, targets = pmd.collate([walk[0], walk[1]])
print("images: a", type(images).__name__, "of", len(images), "tensors")
for image, target, page in zip(images, targets, (example, empty_page)):
    print(f"  {page.image_id}: image {tuple(image.shape)} {image.dtype}, values {image.min():.2f} to {image.max():.2f}")
    print(f"    target boxes  {tuple(target['boxes'].shape)} {target['boxes'].dtype}")
    print(f"    target labels {tuple(target['labels'].shape)} {target['labels'].dtype} -> {target['labels'].tolist()}"
          f" ({[pmd.LABEL_TO_CLASS[int(i)] for i in target['labels']]})")
print("\nFirst target box [x0, y0, x1, y1] in pixels:", [round(float(v), 1) for v in targets[0]["boxes"][0]])

# What the model's own transform does to this batch: normalise, resize, pad into one tensor.
model_preview.to(device)
on_device = [image.to(device) for image in images]
on_device_targets = [{"boxes": t["boxes"].to(device), "labels": t["labels"].to(device)} for t in targets]
with torch.no_grad():
    batch, batch_targets = model_preview.transform(on_device, on_device_targets)
    features = model_preview.backbone(batch.tensors)
print("\nInside the model:")
print("  one padded tensor:", tuple(batch.tensors.shape), "| true image sizes (h, w):", batch.image_sizes)
print("  first target box after the internal resize:", [round(float(v), 1) for v in batch_targets[0]["boxes"][0]])
print("  feature pyramid:", {name: tuple(feature.shape[1:]) for name, feature in features.items()})

**What to notice.** The page went in at 1240 × 1754 and the network works on a padded tensor of about 800 × 1152. The target box was scaled by the same factor. All of this happens inside the model.

### The four losses

In training mode the model returns **losses** instead of detections: four numbers, each teaching a different part of the network.

| Loss | Stage | What it teaches |
| --- | --- | --- |
| `loss_objectness` | RPN | For each sampled anchor: is there a mark here, or background? |
| `loss_rpn_box_reg` | RPN | For positive anchors: how to shift and stretch the anchor to fit the mark |
| `loss_classifier` | Box head | For each sampled proposal: background, exclusion or price change? |
| `loss_box_reg` | Box head | For proposals on a mark: the final, precise box edges |

The training loop adds the four and takes one gradient step on the sum. A page without marks still contributes: all its anchors and proposals are background examples, so the two classification losses are active and the two box losses are zero.

The cell computes the losses once, with no learning. The predictor is new, so the classifier loss starts near ln 3 ≈ 1.1, the loss of a three-way guess. The exact values do not matter here, only that all four are finite numbers.

One detail matters for the next cell. In training mode the model does not use every anchor and every proposal: it draws a random sample of them for each loss. Two measurements of the same weights therefore differ a little unless the random seed is the same. `measure_losses` fixes the seed so that numbers can be compared.

In [ ]:
def measure_losses(network, seed=0):
    """The four losses on the walkthrough batch, without learning; the seed fixes which anchors are sampled."""
    network.train()
    torch.manual_seed(seed)
    with torch.no_grad():          # look, do not learn
        return {name: float(value) for name, value in network(on_device, on_device_targets).items()}


losses_before = measure_losses(model_preview)
display(pd.DataFrame({"loss": list(losses_before), "value before training": list(losses_before.values())}).round(4))
assert all(math.isfinite(v) for v in losses_before.values())
other_sample = sum(measure_losses(model_preview, seed=1).values())
print(f"Sum of the four: {sum(losses_before.values()):.4f}. With another random sample of anchors and proposals: "
      f"{other_sample:.4f}. That difference is sampling noise, not learning.")

model_preview.eval()
with torch.no_grad():
    outputs = model_preview(on_device)
for page, output in zip((example, empty_page), outputs):
    print(f"{page.image_id}: eval mode returns", {k: tuple(v.shape) for k, v in output.items()},
          "| highest score", round(float(output["scores"].max()), 3) if len(output["scores"]) else None)
print("\nThese are saved-weight detections in reload mode; otherwise the new predictor is untrained. Compare with section 8.")

**What to notice.** In evaluation mode the same model returns, per image, a dictionary with `boxes` `[N, 4]`, `labels` `[N]` and `scores` `[N]`, already filtered by the score floor, non-maximum suppression and the per-page cap. Training mode and evaluation mode give different *kinds* of output; that is a torchvision convention worth remembering.

### One real optimisation step

So far the network has only been looked at. Training is the same forward pass with three more lines. In training mode this cell takes **one** real step on the walkthrough batch, so you can see each of them do its work:

1. `backward()` computes, for every trainable weight, how the summed loss would change if that weight grew a little. That number is the weight's **gradient**.
2. `optimiser.step()` moves every trainable weight a small distance against its gradient. The distance is the learning rate times the gradient.
3. `zero_grad()` clears the gradients for the next batch.

The step uses plain SGD at the recipe's learning rate, without warm-up, momentum or weight decay, to keep the arithmetic visible. The real loop in section 7 adds those three. The cell measures the losses again afterwards, on the same batch with the same seed. **Reload mode skips this entire optimization step**, preserving saved weights. This correction pass uses reload mode only.

In [ ]:
if RELOAD_RUN_DIR is None:
    trainable = {name: parameter for name, parameter in model_preview.named_parameters() if parameter.requires_grad}
    frozen = {name: parameter for name, parameter in model_preview.named_parameters() if not parameter.requires_grad}
    watched = ["roi_heads.box_predictor.cls_score.weight", "rpn.head.cls_logits.weight", "backbone.body.layer4.2.conv3.weight"]
    snapshot = {name: parameter.detach().clone() for name, parameter in model_preview.named_parameters()}
    step_optimiser = torch.optim.SGD(list(trainable.values()), lr=cfg.learning_rate)
    
    model_preview.train()
    torch.manual_seed(0)                                   # the same sample of anchors as in measure_losses
    losses = model_preview(on_device, on_device_targets)   # this time WITH gradients
    total = sum(losses.values())
    step_optimiser.zero_grad()
    total.backward()                                       # 1. gradients of the summed loss
    gradient_norms = {name: float(trainable[name].grad.norm()) for name in watched}
    step_optimiser.step()                                  # 2. every trainable weight moves against its gradient
    step_optimiser.zero_grad()                             # 3. ready for the next batch
    
    losses_after = measure_losses(model_preview)
    display(pd.DataFrame({"loss": list(losses_before), "before the step": list(losses_before.values()),
                          "after one step": [losses_after[name] for name in losses_before]}).round(4))
    change = sum(losses_after.values()) - sum(losses_before.values())
    print(f"Sum of the four losses on this batch: {sum(losses_before.values()):.4f} -> {sum(losses_after.values()):.4f} "
          f"({'down' if change < 0 else 'up'} by {abs(change):.4f}; sampling noise was about "
          f"{abs(other_sample - sum(losses_before.values())):.4f}).")
    moved = sum(not torch.equal(parameter.detach(), snapshot[name]) for name, parameter in trainable.items())
    still = sum(torch.equal(parameter.detach(), snapshot[name]) for name, parameter in frozen.items())
    print(f"Trainable tensors that changed: {moved} of {len(trainable)}. Frozen tensors left unchanged: {still} of {len(frozen)} "
          f"(the first backbone stages, e.g. {next(iter(frozen))}).")
    for name in watched:
        shift = float((trainable[name].detach() - snapshot[name]).abs().max())
        print(f"  {name}: gradient norm {gradient_norms[name]:.3f}, largest change of one weight {shift:.5f}")
    if change >= 0:
        print("The loss did not fall. One step at the full learning rate can overshoot; that is what warm-up is for.")
    
    del snapshot, trainable, frozen, step_optimiser, losses, total
else:
    print("Saved-weight reload: optimization walkthrough skipped; no weights were updated.")

del model_preview, on_device, on_device_targets, batch, features, outputs
if device.type == "mps":
    torch.mps.empty_cache()

**What to notice.** The table shows the same batch before and after the step. Read the printed lines under it:

- *The loss moved by far more than the sampling noise.* That is learning, on a batch of two pages. It says nothing about other pages: after one step the network has only become better at *these* two.
- *Only trainable tensors changed.* The frozen backbone stages have no gradient and stay exactly as COCO left them.
- *The new predictor has the largest gradient.* It started from random values, so it has the most to correct. This is why the real loop begins with a warm-up: large first steps on the new layer must not disturb the pretrained ones.

This model is thrown away. The training run below starts again from the COCO weights.

### The recipe

These are the values of the configuration cell, with the reason for each. They follow training specification section 7.3, stage 1 (synthetic pages). **All are proposed.**

| Setting | Value | Why |
| --- | --- | --- |
| Architecture | Faster R-CNN ResNet-50 FPN, COCO-pretrained, 3-class predictor | Specification 7.3 |
| Working resolution | shorter side 800, longer side at most 1333 | Specification default. Native resolution was not compared (one epoch of one run is no evidence); see section 4.6 |
| Anchors | `torchvision_default` | Specification default; keeps every pretrained RPN weight. A wider set, `pen_marks_wide`, fits the box shapes better on paper; the short comparison of section 4.6 was inconclusive |
| Optimiser | SGD, momentum 0.9, weight decay 5e-4 | Specification 7.3. SGD with momentum is the standard choice for this detector family |
| Learning rate | 0.005 | Specification 7.3 |
| Warm-up | linear over the first 100 updates | **Deviation.** The specification says 500 updates, written for 1,500 to 3,000 pages. Here an epoch is 88 updates, so 500 would be more than a quarter of the whole run |
| Schedule | × 0.1 at 70 % and at 90 % of the epochs | Specification 7.3 |
| Batch size | 4 (or 2 with 2 accumulation steps) | Specification 7.3 |
| Epochs | up to 20 | Specification 7.3 |
| Model selection | best validation epoch by mean per-class detection F1 | See below |
| Early stopping | after 5 evaluations without improvement | Specification 7.3 |
| Augmentation | section 4.7; no horizontal flip; **rotation off**; no perspective jitter | **Deviation.** Specification 7.3 lists rotation up to 5 degrees and perspective jitter. Rotation is implemented (`rotation_degrees`) but off here, and perspective jitter is not implemented. See the note below the table |
| Class weighting | none | Specification 7.3; the classes are balanced (section 4.3) |
| Seed | 20260922 | Specification 7.3 |

**Deviations from specification 7.3, in one place** (all proposed, all for this synthetic stage):

- *Warm-up* over 100 updates, not 500 (the run is short).
- *Rotation off.* A box must stay upright, so a rotated mark gets the upright box around its rotated rectangle. For a row strike of 766 × 8 pixels that box is about 35 pixels tall at 2 degrees and about 75 pixels at 5 degrees (width × sin(angle) + height × cos(angle)): the label would stop describing the ink. M4 also deskews a page before M6 sees it. `AUGMENT = replace(pmd.AugmentConfig(), rotation_degrees=...)` turns it on.
- *No perspective jitter.* It belongs with real photographs, upstream of M4's correction.
- *Model selection* by per-class detection F1, not per-row F1 (explained below).

> **Concepts.** An **epoch** is one pass over all training pages. An **update** is one optimiser step on one batch. The **learning rate** is the step size; **warm-up** starts it small so the randomly initialised layers do not wreck the pretrained ones in the first steps; the **schedule** lowers it late in training so the weights settle. **Early stopping** ends the run when validation stops improving, because more training would only memorise the training pages.

**Model selection.** After every epoch the model predicts on the validation pages. For each class the best F1 over all score thresholds (at IoU 0.5) is computed, and the two are averaged. The epoch with the highest value is kept as `best.safetensors`. The specification words this as per-*row* F1; per-class detection F1 is used here because a per-row score depends on the row boxes M5 supplies (section 8.7 shows how strongly), and a detector should not be chosen through another module's output.

The plot shows the learning rate the run will use.

In [ ]:
recipe = pd.DataFrame([{"setting": key, "value (proposed)": str(value)} for key, value in cfg.to_record().items()
                       if key not in {"augment", "artifacts_root"}])
display(recipe.set_index("setting"))

steps_per_epoch = math.ceil(math.ceil(len(train_pages) / cfg.batch_size) / cfg.accumulation_steps)
schedule = [pmt.learning_rate_at(step, step // steps_per_epoch, cfg) for step in range(steps_per_epoch * cfg.epochs)]
fig, axis = plt.subplots(figsize=(9, 2.8))
axis.plot(np.arange(len(schedule)) / steps_per_epoch, schedule, color=PALETTE["price_change"])
for start in pmt.decay_epochs(cfg):
    if start < cfg.epochs:
        axis.axvline(start, color="black", linestyle=":", linewidth=0.8)
axis.set(xlabel="Epoch", ylabel="Learning rate", yscale="log",
         title=f"Planned learning rate: {steps_per_epoch} updates per epoch, warm-up over {cfg.warmup_steps} updates, "
               f"x{cfg.lr_gamma} from epochs {[e + 1 for e in pmt.decay_epochs(cfg) if e < cfg.epochs]}")
save_show(fig, "learning_rate")
print(f"{len(train_pages)} training pages / batch {cfg.batch_size} = {math.ceil(len(train_pages) / cfg.batch_size)} batches per epoch; "
      f"up to {cfg.epochs} epochs.")

### Run it

The next cell trains. On an Apple M4 Max with the default recipe an epoch took about 1.7 to 2 minutes while another job shared the GPU, and the 20-epoch reference run of 2026-10-12 (`penmarks-fasterrcnn-20261012T145839Z-8529778e`) took 37 minutes. Early stopping can end a run sooner.

What it writes under `artifacts/models/penmarks/<run_id>/`:

| File | Content |
| --- | --- |
| `config.json` | The exact configuration |
| `history.jsonl` | One line per epoch: losses, validation scores, learning rate, seconds |
| `best.safetensors` | The weights of the best validation epoch |
| `last.pt` | Network, optimiser and counters after the last finished epoch, for resuming |
| `run.json` | Status, best epoch, data hashes, M4 versions, library versions, code revision |

Each epoch prints one line. **What to watch:** the training loss should fall quickly in the first epochs; the validation F1 should rise and then flatten; `*best*` marks an epoch that improved the selection score.

- **Interrupting** the kernel is safe: the best checkpoint so far is kept and the run is marked `interrupted`.
- **Resuming:** set `RESUME_RUN_DIR` in the configuration cell to the run folder and rerun the notebook. Training continues after the last finished epoch.
- **Skipping training:** set `RELOAD_RUN_DIR` to a finished run to evaluate it without training (section 9).

In [ ]:
if RELOAD_RUN_DIR is not None:
    print("Reloading a saved run; no training:", RUN_DIR)
else:
    started = time.time()
    run = pmt.train(cfg, train_pages, val_pages, data_record=data_record, resume_from=RESUME_RUN_DIR)
    print(f"\nRun {run['run_id']}: {run['status']}, {run['completed_epochs']} epochs, best epoch {run['best_epoch']} "
          f"(validation score {run['best_score']:.4f}); {(time.time() - started) / 60:.1f} minutes in this cell.")

# Always continue from what is on disk: the best epoch, rebuilt and loaded strictly.
model, cfg, run_record = pmt.load_run(RUN_DIR, device=device)
history = pd.DataFrame(pmt.read_history(RUN_DIR))
MODEL_VERSIONS = {"penmark_model": f"penmarks/{RUN_ID}", "penmark_preprocessing": serving.PREPROCESSING_VERSION}
print(f"Loaded the best epoch ({run_record['best_epoch']} of {run_record['completed_epochs']}) from {RUN_DIR.name}/best.safetensors")
note(f"{run_record['completed_epochs']} epoch(s) on {run_record['pages']['train']} pages: the model below is barely trained.")

### Optional: repeat the decision runs

Section 4.6 quoted two short runs that compared the anchor sets at 800 / 1333, and a native-resolution run with the default anchors that was stopped after one epoch. Set `RUN_DECISION_RUNS = True` to run all three arms here: both anchor sets at 800 / 1333, and the default anchors at native resolution. Each arm trains for `DECISION_EPOCHS` epochs into its own run folder, which takes about 10 to 20 minutes per arm (more at native resolution), so this is off by default. It uses the validation split only.

Remember what section 4.6 found: one short run per arm cannot separate differences of a few hundredths. For an answer you can act on, raise `DECISION_EPOCHS` to the full recipe and repeat each arm with several seeds.

In [ ]:
RUN_DECISION_RUNS = False
DECISION_EPOCHS = 4
DECISION_ARMS = {
    "pen_marks_wide @ 800/1333": dict(**pmt.ANCHOR_PRESETS["pen_marks_wide"], min_size=800, max_size=1333),
    "torchvision_default @ 800/1333": dict(**pmt.ANCHOR_PRESETS["torchvision_default"], min_size=800, max_size=1333),
    "torchvision_default @ 1240/1754 (native)": dict(**pmt.ANCHOR_PRESETS["torchvision_default"], min_size=1240,
                                                     max_size=1754, batch_size=2, accumulation_steps=2, eval_batch_size=2),
}
if RUN_DECISION_RUNS and not QUICK_RUN:
    decision_rows = []
    for arm, changes in DECISION_ARMS.items():
        arm_cfg = pmt.TrainingConfig(**{**cfg.to_record(), **changes, "epochs": DECISION_EPOCHS, "run_id": pmt.new_run_id() + "-decision"})
        arm_run = pmt.train(arm_cfg, train_pages, val_pages, data_record=data_record)
        arm_model, _, _ = pmt.load_run(arm_run["run_dir"], device=device)
        arm_predictions = pmt.predict_pages(arm_model, val_pages, device=device, batch_size=arm_cfg.eval_batch_size)
        arm_thresholds = {}
        for mark_class in pmd.TARGET_CLASSES:
            found = pme.class_detections(val_pages, arm_predictions, mark_class)
            arm_thresholds[mark_class] = pme.select_threshold(pme.threshold_table(found["scores"], found["hits"], found["marks"]))
        arm_point = pme.operating_point(val_pages, arm_predictions, arm_thresholds)
        decision_rows.append({"arm": arm, "best epoch": arm_run["best_epoch"], "selection score": arm_run["best_score"],
                              **{f"{r['style']}": f"{r['detected']}/{r['marks']}" for r in arm_point["styles"]},
                              "minutes per epoch": np.mean([h["seconds"] for h in pmt.read_history(arm_run["run_dir"])]) / 60})
        del arm_model
    display(pd.DataFrame(decision_rows).set_index("arm").round(3))
else:
    print("Decision runs not repeated. The recorded comparison is the table in section 4.6.")

## 8. Results on the validation split

Everything in this section is measured on the **validation** pages: pages the network did not train on, from claims it did not train on. They are still synthetic pages of the same layout with the same three handwriting fonts. Read every number below as *synthetic validation evidence*.

### 8.1 Training curves

In [ ]:
best_epoch = int(run_record["best_epoch"])
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs = history["epoch"]
axes[0].plot(epochs, [row["total"] for row in history["train_loss"]], marker="o", color=PALETTE["price_change"], label="training pages")
axes[0].plot(epochs, [row["total"] for row in history["val_loss"]], marker="s", color=PALETTE["exclusion"], label="validation pages")
axes[0].set(xlabel="Epoch", ylabel="Sum of the four losses", title="Total loss")
axes[0].legend()
TERM_COLOURS = dict(zip(pmt.LOSS_TERMS, [PALETTE["exclusion"], PALETTE["price_change"], PALETTE["acceptance_tick"], PALETTE["review_note"]]))
for term in pmt.LOSS_TERMS:
    axes[1].plot(epochs, [row[term] for row in history["train_loss"]], color=TERM_COLOURS[term], label=term)
    axes[1].plot(epochs, [row[term] for row in history["val_loss"]], color=TERM_COLOURS[term], linestyle="--")
axes[1].set(xlabel="Epoch", ylabel="Loss", yscale="log", title="The four losses (solid: training, dashed: validation)")
axes[1].legend(fontsize=8)
axes[2].plot(epochs, history["val_selection_score"], marker="o", color="black", label="selection score (mean of the two)")
for mark_class in pmd.TARGET_CLASSES:
    axes[2].plot(epochs, [np.nan if row[mark_class] is None else row[mark_class] for row in history["val_best_f1"]],
                 marker=".", color=PALETTE[mark_class],
                 label=f"{CLASS_NAMES[mark_class]}: best F1")
axes[2].axvline(best_epoch, color=PALETTE["true_positive"], linestyle=":", label=f"kept epoch ({best_epoch})")
axes[2].set(xlabel="Epoch", ylabel="F1 at IoU 0.5 (validation)", ylim=(-0.02, 1.02), title="What model selection looks at")
axes[2].legend(fontsize=8, loc="lower right")
for axis in axes:
    axis.set_xticks(list(epochs))
save_show(fig, "training_curves")
display(pd.DataFrame({"epoch": epochs, "learning rate": history["learning_rate"],
                      "train loss": [r["total"] for r in history["train_loss"]],
                      "val loss": [r["total"] for r in history["val_loss"]],
                      "val selection score": history["val_selection_score"], "seconds": history["seconds"]}).round(4).set_index("epoch"))
note("")

**How to read the curves.**

- *Left.* Both losses should fall. The training loss is measured on augmented pages while the weights are still changing, the validation loss on clean pages after the epoch, so the two are not measured in quite the same way and either can be the lower one. A validation loss that *rises* steadily while the training loss keeps falling would mean overfitting.
- *Middle.* The classifier loss starts highest because the predictor is new. The two RPN losses show how the region proposals adapt to pen marks.
- *Right.* This is the curve that decides which epoch is kept. Look at the two classes separately: they do not learn at the same speed. The drops in learning rate (section 7) usually show as a step up followed by a flat stretch.

### 8.2 Predictions, and the score threshold

The network returns every box whose score is at least the **score floor** (0.05). That is far more than anyone should act on. To use the detector you pick a **threshold** per class and keep only boxes at or above it.

> **Concepts.** A detection is a **true positive** when it overlaps a not-yet-matched ground-truth mark of its class at IoU 0.5 or more; otherwise it is a **false positive**. A mark that no kept detection matches is a **false negative** (a miss).
> **Precision** = true positives / detections kept: *of what the detector proposes, how much is right?*
> **Recall** = true positives / marks: *of what is there, how much does it find?*
> **F1** is their harmonic mean: high only when both are high.

Raising the threshold removes weak detections: precision goes up, recall goes down. The plots show that trade-off on validation.

In [ ]:
val_predictions = pmt.predict_pages(model, val_pages, device=device, batch_size=cfg.eval_batch_size)
link_config = load_linking_config()          # configs/pipeline/m6_linking.yaml, unchanged
found = {c: pme.class_detections(val_pages, val_predictions, c) for c in pmd.TARGET_CLASSES}
tables = {c: pme.threshold_table(found[c]["scores"], found[c]["hits"], found[c]["marks"]) for c in pmd.TARGET_CLASSES}
THRESHOLD_RULE = ("on validation, per class: take the thresholds whose F1 at IoU 0.5 is within 0.005 of the best, "
                  "and choose the middle of the widest unbroken run of them")
selected = {c: pme.select_threshold(tables[c], fallback=link_config.score_threshold(c)) for c in pmd.TARGET_CLASSES}
configured = {c: link_config.score_threshold(c) for c in pmd.TARGET_CLASSES}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
for axis, mark_class in zip(axes, pmd.TARGET_CLASSES):
    table = tables[mark_class]
    axis.plot(table["threshold"], table["precision"], color=PALETTE["price_change"], label="precision")
    axis.plot(table["threshold"], table["recall"], color=PALETTE["exclusion"], label="recall")
    axis.plot(table["threshold"], table["f1"], color="black", linewidth=2, label="F1")
    axis.axvline(selected[mark_class], color=PALETTE["true_positive"], linewidth=2, label=f"selected here: {selected[mark_class]:.2f}")
    axis.axvline(configured[mark_class], color=PALETTE["row"], linestyle="--", label=f"in m6_linking.yaml: {configured[mark_class]:.2f}")
    axis.set(xlabel="Score threshold", ylim=(-0.02, 1.02),
             title=f"{CLASS_NAMES[mark_class]}: {found[mark_class]['marks']} validation marks, {len(found[mark_class]['scores'])} detections above the floor")
    axis.legend(fontsize=8, loc="lower center")
axes[0].set_ylabel("Precision, recall, F1 at IoU 0.5")
save_show(fig, "threshold_selection")

rows = []
for mark_class in pmd.TARGET_CLASSES:
    table = tables[mark_class]
    for label, value in (("selected here", selected[mark_class]), ("m6_linking.yaml (proposed)", configured[mark_class])):
        index = int(np.abs(table["threshold"] - value).argmin())
        tp, fp, fn = (int(table[k][index]) for k in ("true_positives", "false_positives", "false_negatives"))
        rows.append({"class": mark_class, "threshold": f"{label}: {value:.2f}", "true positives": tp, "false positives": fp,
                     "misses": fn, "precision": f"{tp}/{tp + fp}" if tp + fp else "undefined (no detection)",
                     "recall": f"{tp}/{tp + fn}", "F1": round(float(table["f1"][index]), 3)})
display(pd.DataFrame(rows).set_index(["class", "threshold"]))
note("")

**How to read this.** The rule chooses the middle of the widest run within 0.005 of best F1. A run can be just one grid point: the full reference run's exclusion plateau is **0.90–0.90**. There is no margin on either side. Validation chose the epoch, thresholds and these reported numbers, so the operating point is optimistic.

The table below prints the plateau bounds, recall and false positives at the selected threshold and ±0.02, and the configured alternative. The configured 0.50 preserves the same 53/54 exclusion recall in the reference run but keeps 49 false positives instead of 21. Neither value is automatically adopted; the team chooses that trade-off. Nineteen of the 21 remaining false positives are parts of already-found composite marks (section 8.5), so F1 rewards suppressing a labelling artefact.

The four claim-grouped half-split checks choose thresholds on one half of validation and score the other. They assess threshold sensitivity only: epoch selection already used all validation claims. They are not an independent final evaluation.

In [ ]:
threshold_alternatives = {}
threshold_rows = []
for mark_class in pmd.TARGET_CLASSES:
    plateau = pme.best_f1_run(tables[mark_class])
    print(mark_class, "near-best F1 plateau:", plateau)
    probes = sorted({round(min(0.99, max(0.01, selected[mark_class] + d)), 2) for d in (-0.02, 0, 0.02)} | {configured[mark_class]})
    counts = pme.threshold_counts(tables[mark_class], probes)
    threshold_alternatives[mark_class] = {"plateau": plateau, "selected": selected[mark_class],
        "configured_alternative": configured[mark_class], "counts": counts, "status": "proposed"}
    threshold_rows.extend({"class": mark_class, **row} for row in counts)
display(pd.DataFrame(threshold_rows).set_index(["class", "threshold"]))
half_split = pme.half_split_check(val_pages, val_predictions)
display(pd.DataFrame(half_split).set_index(["mark_class", "seed"]))
print("No configuration was changed. The team chooses between recall margin and false-proposal burden.")

### 8.3 Precision-recall curves and average precision

A threshold gives one precision and one recall. Sweeping the threshold from high to low traces the **precision-recall curve**, which describes the detector independently of any threshold.

> **Concept: average precision (AP).** The area under the precision-recall curve, between 0 and 1. This notebook uses the COCO definition: at each of 101 recall levels (0.00, 0.01, ..., 1.00) take the best precision achieved at that recall or higher, and average. **AP@0.5** counts a detection as correct at IoU 0.5. **AP@[.5:.95]** averages AP over ten IoU requirements from 0.50 to 0.95, so it also rewards *tight* boxes.

`pycocotools` is not installed in this project, so the calculation is implemented in `pme.average_precision` and unit-tested against hand-worked cases. The cell prints the function: it is short, and reading it is the best explanation of the definition.

In [ ]:
print(inspect.getsource(pme.average_precision))
detection = pme.detection_report(val_pages, val_predictions)
fig, axis = plt.subplots(figsize=(6.4, 4.6))
for mark_class in pmd.TARGET_CLASSES:
    curve = detection[mark_class]["curve"]
    ap = detection[mark_class]["ap50"]
    axis.step([0.0, *curve["recall"]], [1.0, *curve["precision"]], where="post", color=PALETTE[mark_class],
              label=f"{CLASS_NAMES[mark_class]} (AP@0.5 = {ap:.3f})" if ap is not None else f"{CLASS_NAMES[mark_class]} (no marks)")
    kept = [i for i, score in enumerate(curve["score"]) if score >= selected[mark_class]]
    if kept:
        axis.plot(curve["recall"][kept[-1]], curve["precision"][kept[-1]], marker="o", markersize=9, markerfacecolor="none",
                  color=PALETTE[mark_class])
axis.set(xlabel="Recall", ylabel="Precision", xlim=(-0.02, 1.02), ylim=(-0.02, 1.05),
         title="Precision-recall at IoU 0.5 (validation)\ncircle = the selected threshold")
axis.legend(loc="lower left")
save_show(fig, "precision_recall")
display(pd.DataFrame([{"class": c, "validation marks": detection[c]["marks"], "detections above the floor": detection[c]["detections"],
                       "AP@0.5": detection[c]["ap50"], "AP@[.5:.95]": detection[c]["ap50_95"]} for c in pmd.TARGET_CLASSES]
                     ).set_index("class").round(3))
note("")

**How to read this.** A perfect detector's curve runs along the top and down the right edge (AP = 1). A curve that drops early means the highest-scoring detections already include mistakes. The circle marks where the selected threshold sits on the curve.

AP@[.5:.95] is always lower than AP@0.5. Here the gap is informative: it is large when the detector finds marks but draws their boxes loosely. Section 8.4 shows where: exclusion boxes are drawn less tightly than price-change boxes, and the thin row strike least tightly of all.

### 8.4 The operating point: counts, per style, box quality

From here on the **selected thresholds** are applied. The first table is the detector's result on validation in counts. The second breaks recall down by style and by handwriting font (the dataset's stand-in for a writer).

In [ ]:
point = pme.operating_point(val_pages, val_predictions, selected)


def fraction(numerator, denominator):
    return f"{numerator}/{denominator} = {numerator / denominator:.3f}" if denominator else f"{numerator}/0 (undefined)"


summary = []
for mark_class, c in point["classes"].items():
    summary.append({"class": mark_class, "threshold": selected[mark_class], "marks": c["marks"],
                    "true positives": c["true_positives"], "false positives": c["false_positives"], "misses": c["false_negatives"],
                    "precision": fraction(c["true_positives"], c["true_positives"] + c["false_positives"]),
                    "recall": fraction(c["true_positives"], c["marks"]),
                    "F1": None if c["f1"] is None else round(c["f1"], 3)})
display(pd.DataFrame(summary).set_index("class"))

style_recall = pd.DataFrame(point["styles"])
style_recall["order"] = [TARGET_STYLES.index((c, s)) for c, s in zip(style_recall["mark_class"], style_recall["style"])]
style_recall = style_recall.sort_values("order")
fig, axes = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [1.5, 1]})
bars = axes[0].bar(range(len(style_recall)), style_recall["recall"].fillna(0),
                   color=[PALETTE[c] for c in style_recall["mark_class"]])
for bar, (_, row) in zip(bars, style_recall.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{row['detected']}/{row['marks']}", ha="center", fontsize=9)
axes[0].set(xticks=range(len(style_recall)), ylim=(0, 1.15), ylabel="Recall at IoU 0.5",
            title="Recall per style at the selected thresholds (found / marks; validation)")
axes[0].set_xticklabels(style_recall["style"], rotation=18, ha="right")
writers = pd.DataFrame(point["writers"])
bars = axes[1].bar(range(len(writers)), writers["recall"].fillna(0), color=PALETTE["row"])
for bar, (_, row) in zip(bars, writers.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{row['detected']}/{row['marks']}", ha="center", fontsize=9)
axes[1].set(xticks=range(len(writers)), ylim=(0, 1.15), title="Recall per handwriting font (all three fonts are also in training)")
axes[1].set_xticklabels(writers["writer"], rotation=10)
save_show(fig, "recall_by_style")
note("")

**How to read this.** The numbers above each bar are *found / marks*. The denominators are small (6 to 24 per style), so read the counts: one more or one fewer miss moves a bar by several points. Orange-red bars are exclusion styles, blue bars price-change styles.

The right-hand chart is **not** a test of new handwriting. All three fonts occur in training as well. It only shows whether one font is harder than the others. How the detector copes with a writer it has never seen cannot be measured with this dataset.

Next: how *good* are the boxes of the marks that were found?

In [ ]:
hits = pd.DataFrame(point["hits"])
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8), gridspec_kw={"width_ratios": [1, 1.4]})
if len(hits):
    for mark_class in pmd.TARGET_CLASSES:
        axes[0].hist(hits.loc[hits["mark_type"] == mark_class, "iou"], bins=np.linspace(0.5, 1.0, 21), alpha=0.65,
                     color=PALETTE[mark_class], label=CLASS_NAMES[mark_class])
    order = [s for _, s in TARGET_STYLES if s in set(hits["style"])]
    groups = [hits.loc[hits["style"] == s, "iou"].to_numpy() for s in order]
    axes[1].boxplot(groups, showfliers=False, medianprops={"color": "black"})
    for index, values in enumerate(groups, start=1):
        axes[1].scatter(np.full(len(values), index) + np.linspace(-0.15, 0.15, len(values)), values, s=10, alpha=0.6,
                        color=PALETTE["price_change"] if order[index - 1].startswith("amount") else PALETTE["exclusion"])
    axes[1].set_xticks(range(1, len(order) + 1))
    axes[1].set_xticklabels(order, rotation=18, ha="right")
    axes[0].legend()
    print("Median IoU of matched boxes:", {c: round(float(np.median(v)), 3) for c, v in point["matched_ious"].items() if v})
else:
    print("No detection matched a mark at these thresholds, so there is no IoU to plot.")
axes[0].set(xlabel="IoU between a true positive and its mark", ylabel="True positives", title="How tight are the boxes that were found?")
axes[1].set(ylabel="IoU", ylim=(0.45, 1.02), title="The same, per style (each dot is one found mark)")
save_show(fig, "matched_iou")

**How to read this.** IoU 1.0 is a pixel-perfect box; 0.5 is the lowest value that still counts as found.

Expect the bare **row strike** to have the lowest IoU. Its box is only 4 to 12 pixels tall. If a predicted box is 2 pixels too tall on an 8-pixel box, IoU already falls to 0.8; 8 pixels too tall, and it is 0.5. IoU punishes thin boxes hard for errors that do not matter downstream: the linker only needs to know *which row* the strike is on. That is why section 8.7 measures what actually matters, the row, without an IoU threshold.

### 8.5 The errors

Counting errors is not enough; look at them. Every false positive is sorted into one kind:

| Kind | Meaning |
| --- | --- |
| `on_distractor` | At least 30 % of the box lies on a tick or a review note |
| `part_of_found_mark` | A component (cross, remark or strike) inside an already-found composite mark |
| `duplicate` | A second box on a mark that a higher-scoring box already found |
| `wrong_class` | On a mark of the other class |
| `poorly_localised` | Touches a mark of its class, but with IoU below 0.5 |
| `background` | None of the above |

Each miss is sorted as well: `below_threshold` (a matching box exists, but its score is under the threshold), `detected_as_other_class`, `poorly_localised`, or `not_detected`.

In [ ]:
fp_summary = point["false_positive_summary"]
print("False positives at the selected thresholds:", fp_summary["total"])
print(f"  on pages that have no mark at all: {fp_summary['on_pages_without_marks']}"
      f"   (pages without marks: {point['pages_without_marks']['pages']}, of which {point['pages_without_marks']['with_a_false_positive']} got a false positive)")
display(pd.DataFrame([{k: fp_summary[k] for k in pme.FALSE_POSITIVE_KINDS}], index=["false positives by kind"]))
distractor_hits = Counter(fp["distractor_class"] for fp in point["false_positives"] if fp["kind"] == "on_distractor")
total_distractors = Counter(d.mark_class for p in val_pages for d in p.distractors)
print("Distractors on the validation pages:", dict(total_distractors), "| false positives on them:", dict(distractor_hits))
miss_kinds = Counter(m["kind"] for m in point["misses"])
print("Misses by kind:", dict(miss_kinds) if miss_kinds else "none")
if point["misses"]:
    display(pd.DataFrame(point["misses"])[["image_id", "mark_class", "style", "kind", "best_score_at_iou"]])

val_by_id = pmd.records_by_id(val_pages)


def error_gallery(items, title, name, colour, caption, limit=8):
    """Crops around false positives or misses, each with its box and a caption."""
    if not items:
        print(f"{title}: none to show.")
        return
    items = items[:limit]
    columns = 4
    rows = math.ceil(len(items) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(15, 1.9 * rows + 0.5), squeeze=False)
    for axis in axes.ravel():
        axis.axis("off")
    for axis, item in zip(axes.ravel(), items):
        page = val_by_id[item["image_id"]]
        window = crop_around(item["box"], page, min_width=300, min_height=110)
        axis.axis("on")
        draw_page(axis, page, rows=False, label=False, crop=window)
        pme.draw_boxes(axis, [item["box"]], colour, pad=4, linewidth=2.2, linestyle="--")
        axis.set_title(caption(item), fontsize=8)
    fig.suptitle(title, fontsize=12)
    save_show(fig, name)


error_gallery(sorted(point["false_positives"], key=lambda fp: -fp["score"]),
              "False positives, highest score first (dashed: the wrong detection; solid/dotted: ground truth)",
              "false_positives", PALETTE["false_positive"],
              lambda fp: f"{fp['image_id']}: {fp['mark_type']} {fp['score']:.2f}\n{fp['kind']}"
                         + (f" ({fp['distractor_class']})" if fp["distractor_class"] else ""))
error_gallery(point["misses"], "Missed marks (dashed: the mark that was not found)", "misses", PALETTE["missed"],
              lambda m: f"{m['image_id']}: {m['mark_class']} / {m['style']}\n{m['kind']}")

**What to look for.** The dataset has many more ticks and notes than target marks, so "false positives on distractors" compared with "distractors on the pages" tells you how well the detector learned to ignore them. A false positive on a page without marks matters most in practice: it is a proposal on a page where the surveyor did nothing.

**A caution about the kinds.** In the recorded run every false positive was an exclusion box, and the eight highest-scoring ones (the gallery) are all an extra box on a *part* of a real exclusion mark: the cross alone, or the remark alone, next to the correct box around both. The cross alone overlaps the whole mark by less than IoU 0.1, so it is filed under `background` although it sits on a real mark. Read `background` as "IoU below 0.1 with every mark", not as "on empty paper". The labels treat a cross and its remark as one mark; the detector also proposes the parts.

### 8.6 Whole pages

The same result, seen the way a reviewer would see it. Green boxes are correct detections, orange-red ones are false positives, and dashed purple ones are marks the detector missed. The first row shows pages where everything was found; the second row shows the pages with the most errors.

In [ ]:
def page_errors(page):
    count = lambda items: sum(item["image_id"] == page.image_id for item in items)   # noqa: E731
    return count(point["false_positives"]) + count(point["misses"])


estimates = [p for p in val_pages if p.page_type == "repair_estimate"]
clean = sorted([p for p in estimates if p.marks and page_errors(p) == 0], key=lambda p: -len(p.marks))[:3]
worst = [p for p in sorted(val_pages, key=lambda p: -page_errors(p)) if page_errors(p) > 0][:3]
gallery = [("all marks found", p) for p in clean] + [("most errors", p) for p in worst]
if gallery:
    fig, axes = plt.subplots(len(gallery), 2, figsize=(16, 3.2 * len(gallery)), squeeze=False,
                             gridspec_kw={"width_ratios": [2, 1]})
    for (axis, key_axis), (kind, page) in zip(axes, gallery):
        counts = pme.draw_detection_page(axis, page_pixels(page), page, point, crop=table_region(page))
        axis.set_title(f"{page.image_id} ({kind}): {counts['hits']} found, {counts['false_positives']} false, {counts['misses']} missed", fontsize=10)
        key_axis.axis("off")
        key_axis.text(0, 0.97, "\n".join(counts["lines"]), va="top", fontsize=9, linespacing=1.4)
    handles = [Rectangle((0, 0), 1, 1, fill=False, edgecolor=PALETTE[k], linestyle=s, label=t) for k, s, t in [
        ("true_positive", "-", "true positive (class and score)"), ("false_positive", "-", "false positive (kind)"),
        ("missed", "--", "missed mark")]]
    fig.legend(handles=handles, loc="outside lower center", ncol=3, frameon=False)
    fig.suptitle("Detections on validation pages at the selected thresholds", fontsize=12)
    save_show(fig, "page_gallery")
else:
    print("No estimate page with marks in this subset.")
note("")

### 8.7 From boxes to rows: the linker

A box on a page is not yet useful. The application needs to know **which printed row** the surveyor marked. That is the job of the linker in `src/claim_cmev/documents/pen_marks/linking.py`. It is rule-based, already tested, and used here **unchanged**, with its configuration `configs/pipeline/m6_linking.yaml` unchanged except for the two score thresholds selected above (applied to an in-memory copy, never written to the file).

How it works, in short:

1. Each row gets a **band**: its box, extended upwards by 0.60 row heights and downwards by 0.25. The band reaches further up because a revised price is usually written above the printed amount.
2. For each mark and row it computes a **link score** from four ingredients: how much of the mark's height lies in the row's band (`v`), how much of the mark lies in the row box (`c`), how much of the row the mark covers (`rc`), and whether the mark's centre is in an allowed column (`col`).
3. **Exclusion (rule EX-1):** link when *exactly one* row reaches a score of 0.50 and leads the runner-up by at least 0.20. Otherwise the mark stays **unlinked** and keeps its candidate rows.
4. **Price change (PC-1, then PC-2):** link when the mark covers at least 30 % of exactly one row's *printed amount* box (and at most 10 % of any other). Otherwise fall back to a stricter version of the score rule.

The linker is deliberately cautious: when two rows are plausible it links to neither. An unlinked mark costs the surveyor one tap; a confidently wrong link could exclude the wrong repair.

#### Oracle rows

The primary oracle row is the **union of its printed cell boxes** (`part_N_quantity/description/action/amount/remarks`), mapped to the corrected frame. Its amount box stays separate. This follows specification step 4 and the M5 parser's union rule. The dataset's full `row_bbox` is a much taller table band, retained only as a contrast.

We compare both with the actual M5 candidate's rows when `M5_ROWS_FILE` is available. The file comes from `LocalLayoutLMv3.predict` → `adapt_predictions` → `pen_mark_row_boxes`. Its schema, validation split and M4 cache hash must match; a quick file is refused in a full execution. True rows absent from M5 stay in the recall denominator.

In [ ]:
eval_config = pme.linking_config_with_thresholds(link_config, selected, label="validation-selected")
print("Linking configuration:", link_config.link_config_version, "->", eval_config.link_config_version, "(in memory only)")
print("  band: above", eval_config.band.extend_above_row_heights, "row heights, below", eval_config.band.extend_below_row_heights)
print("  EX-1: min score", eval_config.link.min_score, "| margin", eval_config.link.margin_to_second,
      "| PC-1: amount overlap >=", eval_config.price_change.amount_overlap_min)

link_example = next((p for p in val_pages if len(p.rows) >= 4 and len(p.marks) >= 2), val_pages[0])
display(pd.DataFrame([{"entry_id": r.entry_id, "page_id": r.page_id, "row_box_norm": [round(v, 4) for v in r.row_box_norm],
                       "amount_box_norm": None if r.amount_box_norm is None else [round(v, 4) for v in r.amount_box_norm]}
                      for r in pme.row_boxes([link_example])]))
print(f"Oracle cell-union RowBox records of {link_example.image_id}; actual M5 rows are compared below.")
m5_rows_path = pme.find_m5_rows_file(M5_ROWS_ROOT) if M5_ROWS_FILE == "auto" else (Path(M5_ROWS_FILE) if M5_ROWS_FILE else None)
m5_rows_record = None
if m5_rows_path is not None:
    m5_rows_record = pme.load_m5_rows(m5_rows_path, split="validation",
        m4_cache_manifest_sha256=pmd._sha256(M4_CACHE_DIR / "manifest.json"), quick_run=QUICK_RUN)
    print("M5 rows file:", shown(m5_rows_path), "| model:", m5_rows_record["m5_model_version"])
else:
    print("No M5 rows file selected/found: actual-M5 comparison is skipped, not a passed check.")

#### Two experiments

The linker is run twice on the validation pages:

- on the **detector's** boxes (what the pipeline would really do), and
- on the **ground-truth mark boxes**, as if a perfect detector had found every mark with score 1 ("oracle detections").

The second run shows what the linker loses by itself. The difference between the two is what the detector loses.

Every ground-truth mark ends in exactly one of five outcomes:

| Outcome | Meaning |
| --- | --- |
| `linked_correct` | Found, and linked to the row the surveyor marked |
| `linked_wrong` | Found, but linked to another row |
| `unlinked_candidate` | Found, left unlinked, and the correct row is among the candidates shown to the surveyor |
| `unlinked_no_candidate` | Found, left unlinked, and the correct row is not among the candidates |
| `not_detected` | No detection of the right class matched it at IoU 0.5 |

In [ ]:
oracle = pme.oracle_predictions(val_pages)
linked = pme.link_pages(val_pages, val_predictions, eval_config, versions=MODEL_VERSIONS)
linked_oracle = pme.link_pages(val_pages, oracle, eval_config, versions={"penmark_model": "oracle-detections"})
outcomes = pme.linking_outcomes(val_pages, val_predictions, linked)
outcomes_oracle = pme.linking_outcomes(val_pages, oracle, linked_oracle)

OUTCOME_COLOURS = {"linked_correct": PALETTE["true_positive"], "linked_wrong": PALETTE["false_positive"],
                   "unlinked_candidate": PALETTE["amount"], "unlinked_no_candidate": PALETTE["review_note"],
                   "not_detected": PALETTE["row"]}


def outcome_frame(rows):
    frame = pd.DataFrame(pme.outcome_summary(rows))
    if len(frame):
        frame["order"] = [TARGET_STYLES.index((c, s)) for c, s in zip(frame["mark_class"], frame["style"])]
        frame = frame.sort_values("order").drop(columns="order")
    return frame


frames = {"oracle detections + linker": outcome_frame(outcomes_oracle), "detector + linker": outcome_frame(outcomes)}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
for axis, (title, frame) in zip(axes, frames.items()):
    bottom = np.zeros(len(frame))
    for outcome in pme.OUTCOMES:
        values = frame[outcome].to_numpy() if len(frame) else np.zeros(0)
        axis.bar(range(len(frame)), values, bottom=bottom, color=OUTCOME_COLOURS[outcome], label=outcome)
        for index, value in enumerate(values):
            if value:
                axis.text(index, bottom[index] + value / 2, str(int(value)), ha="center", va="center", fontsize=8)
        bottom += values
    axis.set(xticks=range(len(frame)), title=f"{title}: what happened to each validation mark")
    axis.set_xticklabels(frame["style"] if len(frame) else [], rotation=18, ha="right")
axes[0].set_ylabel("Ground-truth marks")
axes[1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))
save_show(fig, "linking_outcomes")
for title, frame in frames.items():
    print(title)
    display(frame.set_index(["mark_class", "style"]) if len(frame) else frame)
note("")

**How to read this.** Each bar is one style; its segments add up to the number of validation marks of that style. Green is the good outcome. Start with the **left** panel: with perfect boxes, anything that is not green is the linker's doing. Then compare with the right panel: grey (`not_detected`) is what the detector adds.

Yellow (`unlinked_candidate`) means the mark was found and the correct row was retained as a candidate, but the score or uniqueness requirement failed. It does not necessarily mean two rows qualified. The cell after the headline numbers prints the exact reason and scores.

#### The headline numbers: per-row precision and recall

The M6 specification states its targets per **row**, before any human correction. This notebook uses the following definition (proposed, because the specification does not spell one out):

- A row is **marked** with a class when the ground truth puts a mark of that class on it.
- A row is **flagged** with a class when the pipeline linked a pending mark of that class to it.
- **Precision** = flagged rows that are marked / flagged rows. **Recall** = marked rows that are flagged / marked rows.
- An **unlinked** mark flags no row. It waits for the surveyor, and is counted separately.

No IoU threshold is involved: a strike that was boxed loosely but linked to the right row counts in full.

In [ ]:
def row_table(scores):
    rows = []
    for mark_class in pmd.TARGET_CLASSES:
        s = scores[mark_class]
        rows.append({"class": mark_class, "marked rows": s["marked_rows"], "flagged rows": s["flagged_rows"],
                     "precision": fraction(s["true_positives"], s["flagged_rows"]),
                     "recall": fraction(s["true_positives"], s["marked_rows"]),
                     "wrongly flagged rows": s["false_positives"], "marked rows not flagged": s["false_negatives"]})
    return pd.DataFrame(rows).set_index("class")


row_scores = pme.row_level_scores(val_pages, linked)
row_scores_oracle = pme.row_level_scores(val_pages, linked_oracle)
m4_unreadable = sorted(p.image_id for p in val_pages if p.page_status == "unreadable")
linked_no_rows = pme.link_pages(val_pages, val_predictions, eval_config, versions=MODEL_VERSIONS, pages_without_rows=m4_unreadable)
row_scores_no_rows = pme.row_level_scores(val_pages, linked_no_rows)

linking_comparison = pme.linking_by_rows(val_pages, val_predictions, eval_config,
    versions=MODEL_VERSIONS, m5_rows=m5_rows_record)
comparison = pd.DataFrame(pme.linking_table(linking_comparison))
comparison["recall (count)"] = [fraction(n, d) for n, d in zip(comparison.true_positives, comparison.marked_rows)]
comparison["precision (count)"] = [fraction(n, d) for n, d in zip(comparison.true_positives, comparison.flagged_rows)]
print("Synthetic validation: identical detections and linker, three row sources; before human correction")
print("Unlinked-mark and conflicting-row columns are totals per experiment, repeated across its two classes.")
display(comparison.set_index(["rows", "detections", "mark_class"])[
    ["rows_given", "recall (count)", "precision (count)", "unlinked_marks", "conflicting_rows"]])
if m5_rows_record is not None:
    m5_pages, m5_row_summary = pme.with_m5_rows(val_pages, m5_rows_record)
    print("Actual M5 row mapping:", m5_row_summary)
print(f"Cell-union contrast with no rows on M4-unreadable pages: {m4_unreadable}")
display(row_table(row_scores_no_rows))
print("Proposed marks with rows withheld:", row_scores_no_rows["marks"])
note("")

**How to read this.** Geometry changes the result. The reference run flags exclusion rows at **16/54** with table bands, **49/54** with printed-cell unions, and **33/54** with real M5 rows. Price-change recalls are **46/46**, **46/46** and **43/46** respectively; precision is 100% in all six cases. With oracle detections the exclusion counts are 12, 49 and 30 of 54. Oracle detection scores are not a ceiling: component detections can link where the whole composite mark does not.

Real M5 supplies 192 of 206 rows. Fourteen rows are on two pages M4 could not read; other M5 boxes are taller where assigned OCR text extends the row. Missing rows keep their marked-row denominators and cannot be flagged. The measured M4 → M5 → M6 chain is for **one candidate on synthetic validation**. The table-band result does not justify blaming or tuning the linker. First establish M5's supplied geometry, then assess whether its band needs tuning.

These are **not** the specification's acceptance numbers. The targets (exclusion precision and recall at least 0.90, price change at least 0.85) are defined on held-out real team-marked pages, which do not exist yet.

#### Why does the linker leave marks unlinked?

The next cell picks one exclusion that stayed unlinked *even with a perfect box* and shows the linker's own scores for it.

In [ ]:
unlinked = [row for row in outcomes_oracle if row["mark_class"] == "exclusion" and row["outcome"].startswith("unlinked")]
if unlinked:
    case = unlinked[0]
    page = val_by_id[case["image_id"]]
    item = next(i for i in linked_oracle if page in i.pages)
    scores = pd.DataFrame([{"row": c.entry_id.split(":")[1], "link_score": c.link_score, "v (in band)": c.v, "c (in row box)": c.c,
                            "rc (row covered)": c.rc, "col": c.col, "amt": c.amt}
                           for c in item.result.candidates if c.mark_id == case["pen_mark_id"]])
    truth = next(m for m in page.marks if m.mark_id == case["mark_id"])
    print(f"{case['mark_id']} ({case['style']}), drawn on {case['true_entry_id'].split(':')[1]}: outcome {case['outcome']}, "
          f"reason {case['link_reason']}, rule {case['rule_id']}")
    display(scores.set_index("row"))
    qualifying = int((scores["link_score"] >= eval_config.link.min_score).sum())
    print(f"Rows reaching the minimum score {eval_config.link.min_score}: {qualifying}. EX-1 links only when exactly one does.")

    fig, axis = plt.subplots(figsize=(13, 3.6))
    axis.imshow(page_pixels(page))
    above, below = eval_config.band.extend_above_row_heights, eval_config.band.extend_below_row_heights
    candidates = [r for r in page.rows if r.row_id in set(scores["row"])][:2]
    for row, colour in zip(candidates, (PALETTE["true_positive"], PALETTE["amount"])):
        height = row.box[3] - row.box[1]
        axis.add_patch(Rectangle((row.box[0], row.box[1] - above * height), row.box[2] - row.box[0], height * (1 + above + below),
                                 facecolor=colour, alpha=0.22, edgecolor=colour, linewidth=1.2,
                                 label=f"band of {row.row_id} (row box extended {above} up, {below} down)"))
        pme.draw_boxes(axis, [row.box], colour, linewidth=1.6)
    pme.draw_boxes(axis, [truth.box], PALETTE["exclusion"], pad=3, linewidth=2, labels=[f"the mark ({truth.style})"])
    window = crop_around(truth.box, page, min_width=1000, min_height=260)
    axis.set(xlim=(window[0], window[2]), ylim=(window[3], window[1]), xticks=[], yticks=[],
             title=f"{page.image_id}: {case['link_reason']} with printed-cell union rows")
    axis.legend(loc="lower left", fontsize=8)
    save_show(fig, "unlinked_example")
else:
    print("With oracle detections every exclusion was linked on these pages; there is no unlinked example to show.")

**What this shows.** Read the printed candidate scores and reason for the selected cell-union example. A row's band is scaled by that row box's height. The full table bands are 79 pixels tall; primary printed-cell unions have a median height of 25 pixels in the reference validation set. The old broad-band explanation cannot be carried over to these narrower rows. In the full reference example, SYN-00180_P2 part_5 has just one retained candidate, but its score is about 0.45, below 0.50: vertical overlap alone contributes, while the mark sits outside the printed-cell union horizontally. No row qualifies. Despite its name, `mark_between_rows` here does not mean two rows scored above threshold.

The following sensitivity plot is recomputed on **printed-cell unions**. It changes an in-memory value only. It cannot decide the production band until the geometry supplied by M5 is established.

In [ ]:
extensions = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
sweep = []
for value in extensions:
    varied = eval_config.with_overrides({"band": {"extend_above_row_heights": value}})
    for source, predictions in (("oracle detections", oracle), ("detector", val_predictions)):
        scores = pme.row_level_scores(val_pages, pme.link_pages(val_pages, predictions, varied, versions={"penmark_model": "sensitivity"}))
        for mark_class in pmd.TARGET_CLASSES:
            sweep.append({"extend_above": value, "boxes": source, "class": mark_class, "recall": scores[mark_class]["recall"],
                          "precision": scores[mark_class]["precision"], "true_positives": scores[mark_class]["true_positives"],
                          "marked_rows": scores[mark_class]["marked_rows"]})
sweep = pd.DataFrame(sweep)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for axis, mark_class in zip(axes, pmd.TARGET_CLASSES):
    for source, style in (("oracle detections", "--"), ("detector", "-")):
        part = sweep[(sweep["class"] == mark_class) & (sweep["boxes"] == source)]
        axis.plot(part["extend_above"], part["recall"].astype(float), linestyle=style, marker="o", color=PALETTE[mark_class],
                  label=f"{source}: per-row recall")
    axis.axvline(link_config.band.extend_above_row_heights, color="black", linestyle=":", label="value in m6_linking.yaml")
    axis.set(xlabel="band.extend_above_row_heights", ylim=(-0.02, 1.02), title=f"{CLASS_NAMES[mark_class]}")
    axis.legend(fontsize=8, loc="lower left")
axes[0].set_ylabel("Per-row recall (validation, printed-cell unions)")
fig.suptitle("Sensitivity only: one linker value varied on validation. The configuration is not changed.", fontsize=11)
save_show(fig, "linking_sensitivity")
display(sweep[sweep["boxes"] == "detector"].pivot(index="extend_above", columns="class", values="recall").round(3))

**How to read this.** The dotted line is the configured value. On the full reference validation set both curves are flat: cell-union exclusion recall remains 49/54 and price-change recall 46/46 across extensions 0.0–0.6. This probe offers no evidence that reducing the upward band improves linking for these primary rows. Price changes use the printed-amount box through PC-1.

Do not pick a value from this plot and call it tuned. It is one layout, with oracle rows, and the upward band exists for a reason that this dataset does not exercise (a new price written *above* the printed amount; here it is written in the remarks column).

#### Marks and their rows on the page

Three validation pages with the pipeline's output: each proposed mark is drawn with a line to the row it was linked to. Green is a correct link, orange-red a wrong one, yellow an unlinked mark (waiting for the surveyor).

In [ ]:
outcome_of = {row["pen_mark_id"]: row["outcome"] for row in outcomes if row["pen_mark_id"]}


def draw_links(axis, page, item):
    """One page with every proposed mark and a line to the row it was linked to."""
    axis.imshow(page_pixels(page))
    rows = {r.entry_id: r for r in page.rows}
    for row in page.rows:
        pme.draw_boxes(axis, [row.box], PALETTE["row"], linewidth=0.6)
        axis.text(row.box[0] + 6, (row.box[1] + row.box[3]) / 2, row.row_id, fontsize=7, va="center", color="black",
                  bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.7, "pad": 0.4})
    counts = Counter()
    legend_lines, label_positions = [], []
    for mark in item.result.marks:
        if mark.page_id != page.page_id:
            continue
        box = np.array(mark.box_norm) * [page.width, page.height, page.width, page.height]
        # A component false positive can still link to the correct row. Colour row correctness,
        # not whether its box was matched at IoU 0.5 in the detection metric.
        correct_row = any(m.entry_id == mark.entry_id and m.mark_class == mark.mark_type for m in page.marks)
        outcome = ("linked_correct" if correct_row else "linked_wrong") if mark.entry_id else "unlinked_candidate"
        colour = OUTCOME_COLOURS.get(outcome, PALETTE["amount"] if mark.entry_id is None else PALETTE["false_positive"])
        counts[outcome] += 1
        number = len(legend_lines) + 1
        pme.draw_boxes(axis, [box], colour, pad=4, linewidth=2)
        label_y = (box[1] + box[3]) / 2
        while any(abs(label_y - used) < 18 for used in label_positions):
            label_y += 18
        label_positions.append(label_y)
        axis.text(box[0] - 8, label_y, str(number), fontsize=9, ha="right", color=colour,
                  bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 0.5})
        legend_lines.append(f"{number}. {mark.mark_type} {mark.detection_confidence:.2f} → "
                            + (mark.entry_id.split(':')[1] if mark.entry_id else f"unlinked ({mark.link_reason})"))
        if mark.entry_id in rows:
            target = rows[mark.entry_id].box
            axis.annotate("", xy=(target[0] + 40, (target[1] + target[3]) / 2), xytext=((box[0] + box[2]) / 2, (box[1] + box[3]) / 2),
                          arrowprops={"arrowstyle": "->", "color": colour, "linewidth": 1.4})
    window = table_region(page)
    axis.set(xlim=(window[0], window[2]), ylim=(window[3], window[1]), xticks=[], yticks=[])
    return counts, legend_lines


link_pages_shown = sorted([p for p in val_pages if len(p.marks) >= 2], key=lambda p: -len(p.marks))[:3]
if link_pages_shown:
    fig, axes = plt.subplots(len(link_pages_shown), 2, figsize=(16, 3.4 * len(link_pages_shown)),
                             squeeze=False, gridspec_kw={"width_ratios": [2.3, 1]})
    for (axis, key_axis), page in zip(axes, link_pages_shown):
        counts, legend_lines = draw_links(axis, page, next(i for i in linked if page in i.pages))
        axis.set_title(page.image_id + ": printed-cell unions", fontsize=11)
        key_axis.axis("off")
        key_axis.text(0, 0.95, "\n".join(legend_lines), va="top", fontsize=9, linespacing=1.5)
    fig.suptitle("Proposed marks and row links (synthetic validation; every mark pending)", fontsize=12)
    save_show(fig, "links_on_pages")

### 8.8 What the pipeline hands over: pending marks

The linker's output is a list of `PenMark` records, the contract record the application stores. Look at the columns:

- `state` is **`pending`** for every mark. Detection never produces anything else.
- `detection_confidence` is the detector's score. It is not a probability that the mark is real.
- `entry_id` is the linked row, or empty with a `link_reason` and the `candidate_entry_ids` kept for the surveyor.
- `confirmed_amount` is empty. **The detector does not read handwriting.** A `price_change` mark says "there is a handwritten amount here, for this row". The number itself is typed by the surveyor.

In [ ]:
all_marks = [mark for item in linked for mark in item.result.marks]
assert all(mark.state == "pending" for mark in all_marks), "detection must only ever produce pending marks"
assert all(mark.confirmed_amount is None and mark.origin == "detector" for mark in all_marks)
print(f"{len(all_marks)} marks proposed on {len(val_pages)} validation pages; states: {dict(Counter(m.state for m in all_marks))}")

shown_item = max(linked, key=lambda i: len(i.result.marks))
display(pd.DataFrame([{"mark_id": m.mark_id[:12] + "...", "mark_type": m.mark_type, "detection_confidence": round(m.detection_confidence, 3),
                       "entry_id": m.entry_id, "candidate_entry_ids": [c.split(":")[1] for c in m.candidate_entry_ids],
                       "link_reason": m.link_reason, "rule_id": m.rule_id, "state": m.state,
                       "confirmed_amount": m.confirmed_amount} for m in shown_item.result.marks]))
if shown_item.result.marks:
    print("versions stamped on every mark:", shown_item.result.marks[0].versions)
    print("provenance:", shown_item.result.marks[0].provenance.model_dump())

**What the surveyor still does.** For each pending mark, in the review screen (M9):

- **confirm** it, or **reject** it (a rejected proposal is kept, visibly dismissed);
- **relink** it when it is unlinked or linked to the wrong row, choosing from the candidates or any row;
- for a price change, **type the revised amount**: confirming a price change is impossible without an exact amount;
- **add** a mark the detector missed.

Until then the project's rules hold the line. A pending exclusion does *not* remove the row from the checks, it withholds them. A pending price change leaves the row's price **unresolved**: the system never falls back to the printed amount.

The cell below demonstrates the last rule with the project's own functions. It takes one linked price-change mark, asks for the row's effective price before and after a (simulated) surveyor confirmation. The typed amount comes from the dataset's ground truth, playing the role of the person. It is not something the model produced.

In [ ]:
price_marks = [(item, mark) for item in linked for mark in item.result.marks
               if mark.mark_type == "price_change" and outcome_of.get(mark.mark_id) == "linked_correct"]
if price_marks:
    item, mark = price_marks[0]
    image_id = next(p.image_id for p in item.pages if p.page_id == mark.page_id)
    with (DATASET_DIR / "annotations.jsonl").open() as stream:       # a validation page's own ground truth
        record = next(r for r in map(json.loads, stream) if r["image_id"] == image_id)
    decision = next(r for r in record["row_decisions"] if f"{image_id}:{r['row_id']}" == mark.entry_id)
    printed, typed = decision["declared_amount"], decision["surveyor_revised_amount"]
    before = effective_price_for(printed, item.result.marks, entry_id=mark.entry_id)
    print(f"Row {mark.entry_id}: printed amount {printed}; the model proposed a price-change mark (score {mark.detection_confidence:.2f}).")
    print("  while the mark is pending  ->", before)
    action = MarkAction(action_type="confirm_mark", action_id="act-notebook-demo-1", mark_id=mark.mark_id,
                        confirmed_amount=typed, confirmed_currency="SGD", confirmed_cost_basis=COST_BASIS)
    after_marks = apply_mark_action(list(item.result.marks), action, actor="surveyor:notebook-demo",
                                    recorded_at=datetime.now(timezone.utc), review_revision=1,
                                    rows=pme.row_boxes(item.pages))
    confirmed = next(m for m in after_marks if m.mark_id == mark.mark_id)
    after = effective_price_for(printed, after_marks, entry_id=mark.entry_id)
    print(f"  the surveyor confirms and types {typed} ->", after)
    print("  the confirmed mark:", {"state": confirmed.state, "confirmed_amount": confirmed.confirmed_amount, "decided_by": confirmed.decided_by})
    assert before.effective_price is None and after.effective_price == typed
else:
    print("No correctly linked price-change mark in this run to demonstrate with.")

**What to notice.** Before the confirmation the effective price is `None` with the reason `price_change_pending`, not the printed amount. After it, the price is the typed amount with the source `surveyor_entry`. The printed amount, the surveyor's amount and (elsewhere) the final approval stay three separate values.

### 8.9 The validation report

One function, `pme.evaluate_pages`, produces everything above as a single JSON report: detection, the operating point, linking with the detector's boxes and with oracle boxes. The same function is used for the final evaluation in section 11, so validation and test are scored by identical code. The report is saved next to the figures.

In [ ]:
val_report = pme.evaluate_pages(val_pages, val_predictions, selected, link_config, versions=MODEL_VERSIONS,
                                threshold_label="validation-selected", m5_rows=m5_rows_record)
val_report.update(run_id=RUN_ID, quick_run=QUICK_RUN, threshold_rule=THRESHOLD_RULE, provenance="synthetic",
                  best_epoch=int(run_record["best_epoch"]), threshold_alternatives=threshold_alternatives,
                  half_split_check=half_split)
(EVAL_DIR / "validation_report.json").write_text(json.dumps(val_report, indent=1))
(EVAL_DIR / "thresholds.json").write_text(json.dumps({"score_threshold": selected, "status": "proposed",
                                                     "selected_on": "validation", "rule": THRESHOLD_RULE}, indent=1))
assert val_report["linking"]["oracle_cell_union_rows"]["detector"]["row_level"] == row_scores, "the report and the cells above use the same code"
print("Saved:", shown(EVAL_DIR / "validation_report.json"))
print("Report sections:", [key for key in val_report])

## 9. Reload a saved run without retraining

Training is the slow part. Everything after it can be repeated from the files of a run:

1. Restart the kernel.
2. In the configuration cell set `RELOAD_RUN_DIR = ROOT / "artifacts/models/penmarks/<run_id>"`.
3. Run the notebook. The training cell is skipped, and the run's own saved configuration is used.

`pmt.load_run` rebuilds the network from `config.json` and loads `best.safetensors` **strictly**: every tensor must be present with the right shape. This is a reload for evaluation; it is not the same as resuming training (`RESUME_RUN_DIR`), which also restores the optimiser.

The cell below checks reload consistency on a few validation pages. In reload mode both copies share a builder, so this alone cannot detect a builder error. Section 10 uses an independently constructed torchvision reference and compares validation metrics with the live-network history at the saved best epoch.

In [ ]:
reloaded, reloaded_cfg, reloaded_record = pmt.load_run(RUN_DIR, device=device)
sample_pages = val_pages[:8]
again = pmt.predict_pages(reloaded, sample_pages, device=device, batch_size=cfg.eval_batch_size)
largest, same_counts, compared = 0.0, True, 0
for page in sample_pages:
    first, second = val_predictions[page.image_id], again[page.image_id]
    same_counts &= len(first.scores) == len(second.scores) and np.array_equal(first.labels, second.labels)
    if same_counts and len(first.scores):
        largest = max(largest, float(np.abs(first.boxes - second.boxes).max()))
        compared += len(first.scores)
print(f"Reloaded {reloaded_record['run_id']} (best epoch {reloaded_record['best_epoch']}, status {reloaded_record['status']}).")
print(f"Detections on {len(sample_pages)} pages: same count and classes: {same_counts}; "
      f"largest box difference over {compared} detections: {largest:.5f} px")
assert same_counts and largest < 0.01
del reloaded

## 10. Serving-path check: is the exported model the same model?

The application does not read a training run. It reads a **registry entry**: a folder with the weights and the records that describe them, loaded by `load_penmark_detector` in `src/claim_cmev/documents/pen_marks/detector.py`.

This section exports the run to a throwaway registry in a temporary folder and loads it back **with that same loader**. (The real export is section 12 and is opt-in.) Four things are checked:

1. The entry is complete, and the loader accepts it.
2. The served detector returns the **same detections** as an independently constructed torchvision reference for default COCO anchors. Other anchors use the serving builder and are explicitly a weaker check. Metrics are also compared with the live training history at the saved best epoch.
3. The loader **refuses** an entry whose records were changed.
4. The served detections flow through the linker to the same pending marks.

> **Why "it loads" proves nothing.** A weight file does not contain everything that defines a network. The anchor shapes, the resize limits, the score floor and the batch-norm `eps` are constructor settings; they are not tensors and are not in the file. A network built with different settings can load the same weights *without any error* and give different boxes. So the entry records those settings in `preprocessing.json`, the loader builds the network from them, and the check compares actual detections.

In [ ]:
check_registry = tempfile.TemporaryDirectory(prefix="penmarks-serving-check-")
check_entry = pmt.export_for_serving(RUN_DIR, f"{RUN_ID}-check", registry_root=check_registry.name, evaluation_report=val_report,
                                     thresholds=selected, check_pages=val_pages[:8], created_by=CREATED_BY, device=device,
                                     threshold_rule=THRESHOLD_RULE, threshold_alternatives=threshold_alternatives,
                                     pages_used=pmt.pages_used(run_record, {"train": train_pages_all, "validation": val_pages_all}),
                                     quick_run=QUICK_RUN)
print("Entry files:")
for path in sorted(check_entry.iterdir()):
    print(f"  {path.name:26s} {path.stat().st_size:>12,} bytes")
check_manifest = json.loads((check_entry / "manifest.json").read_text())
display(pd.DataFrame([{"field": k, "value": str(check_manifest[k])[:110]} for k in (
    "model_id", "version", "status", "task", "architecture", "base_checkpoint", "base_checkpoint_licence", "weights_sha256",
    "label_schema_version", "preprocessing_version", "split_version", "seed", "provenance", "contract_version")]).set_index("field"))
print("preprocessing.json (what the served detector applies):")
print(json.dumps(json.loads((check_entry / "preprocessing.json").read_text()), indent=1))
print("\nserving_check.json, written by the export after the loader reproduced the run's detections:")
print(json.dumps(json.loads((check_entry / "serving_check.json").read_text()), indent=1))

**What to look for.** `status` is `candidate` and `provenance` is `synthetic`. `preprocessing.json` records the colour order (`RGB`), the value range (0-1), the resize limits, the anchors, `frozen_batchnorm_eps`, and the score floor, NMS IoU and per-page cap. `serving_check.json` is the export's own proof: the entry was loaded with the serving loader *before* the folder was moved into place, and its detections were compared with the run's.

Now load the entry the way the application would and compare on **all** validation pages. The detector takes the corrected page as an **RGB** `uint8` array.

> **The BGR/RGB boundary.** M4 works with OpenCV, whose arrays are in **BGR** order. The detector was trained on RGB. The stored `rectified.png` decodes to RGB with Pillow, which is what this notebook does. A caller that holds an OpenCV array must convert it first (`image[..., ::-1]`). The detector cannot notice a swapped order: a BGR page is simply a different picture, with blue ink turned brown.

In [ ]:
detector = load_penmark_detector(check_entry, device=str(device))
print("Loaded:", detector.model_version, "| status:", detector.status, "| versions for marks:", detector.versions)

independent, reference_info = pmt.reference_model(RUN_DIR, device=device)
independent_predictions = pmt.predict_pages(independent, val_pages, device=device, batch_size=cfg.eval_batch_size)
print("Reference construction:", reference_info)
history_comparison = pmt.history_check(RUN_DIR, val_pages, independent_predictions)
print("Live-network history comparison:", history_comparison)
assert not history_comparison["ran"] or history_comparison["passed"]
del independent
served, box_difference, score_difference, same = {}, 0.0, 0.0, True
for page in val_pages:
    boxes, labels, scores = detector.predict(page_pixels(page))
    served[page.image_id] = pme.PagePrediction(boxes, labels, scores)
    reference = independent_predictions[page.image_id]
    if len(scores) != len(reference.scores) or not np.array_equal(labels, reference.labels):
        same = False
    elif len(scores):
        box_difference = max(box_difference, float(np.abs(boxes - reference.boxes).max()))
        score_difference = max(score_difference, float(np.abs(scores - reference.scores).max()))
total = sum(len(p.scores) for p in served.values())
print(f"\nServed detector vs independent reference, {len(val_pages)} validation pages, {total} detections above the floor, device {device}:")
print(f"  same number of detections and same classes on every page: {same}")
print(f"  largest box difference: {box_difference:.6f} px   (tolerance {pmt.BOX_TOLERANCE_PX} px)")
print(f"  largest score difference: {score_difference:.2e}   (tolerance {pmt.SCORE_TOLERANCE})")
assert same and box_difference <= pmt.BOX_TOLERANCE_PX and score_difference <= pmt.SCORE_TOLERANCE

# The public method returns contract records in the 0-1 frame of the corrected page.
records = detector.detect(page_pixels(link_example), page_id=link_example.page_id)
print(f"\ndetect() on {link_example.image_id}: {len(records)} Detection records; the three with the highest score:")
for record in records[:3]:
    print("  ", record.model_dump())

# Downstream: the served detections through the unchanged linker.
linked_served = pme.link_pages(val_pages, served, eval_config, versions=detector.versions)
served_scores = pme.row_level_scores(val_pages, linked_served)
print("\nPer-row scores from the served detector equal those of section 8.7:",
      all(served_scores[c] == row_scores[c] for c in pmd.TARGET_CLASSES))
assert all(m.state == "pending" for item in linked_served for m in item.result.marks)
assert all(served_scores[c] == row_scores[c] for c in pmd.TARGET_CLASSES)

**How to read this.** On the same device the served detector and the training-side model agree exactly or to within rounding. On another device (the loader's default is the CPU) the same weights give boxes that differ by small fractions of a pixel, because the arithmetic is ordered differently; that is expected and harmless.

### The `eps` trap, demonstrated

The pretrained backbone uses *frozen* batch normalisation with `eps=0.0`; a fresh constructor defaults to `1e-5`. This value is absent from the state dict. A strict load can therefore pass while detections differ. The cell compares **all validation pages**, after the selected thresholds, pairing same-class boxes at IoU ≥ 0.9. It reports unmatched boxes appearing or vanishing and the largest score shift among matched pairs. This measures output differences without confusing the order of detections.

In [ ]:
from safetensors.torch import load_file
from torchvision.ops.misc import FrozenBatchNorm2d

careless = serving.build_model(detector.settings)
for module in careless.modules():
    if isinstance(module, FrozenBatchNorm2d):
        module.eps = 1e-5                      # what you get if you forget to restore it
result = careless.load_state_dict(load_file(str(check_entry / serving.WEIGHTS_FILE)), strict=True)
print("Strict load into the careless network:", result)
careless_predictions = pmt.predict_pages(careless.to(device), val_pages, device=device, batch_size=cfg.eval_batch_size)
eps_comparison = pme.compare_detection_sets(val_pages, independent_predictions, careless_predictions, selected, iou_threshold=0.9)
print(json.dumps(eps_comparison, indent=2))
(EVAL_DIR / "eps_comparison.json").write_text(json.dumps(eps_comparison, indent=2))
print("The loader restores frozen_batchnorm_eps:", detector.settings.frozen_batchnorm_eps)
del careless

### The loader refuses a changed entry

A worker must refuse to start with a model it cannot verify. A refusal is a start-up error, never a claim result. The hash chain detects accidental edits; someone who changes both a record and its manifest hash can bypass it. Setting `status: approved` by hand is also not prevented: approval must be a recorded human decision outside these files. The cell makes three damaged copies of the entry's records and shows the loader's answer to each.

In [ ]:
import shutil


def damaged_copy(name, edit):
    """A copy of the entry (weights linked, not copied) with one record changed by `edit`."""
    copy = Path(check_registry.name) / "damaged" / name
    copy.mkdir(parents=True)
    for path in check_entry.iterdir():
        if path.name == serving.WEIGHTS_FILE:
            os.symlink(path, copy / path.name)
        else:
            shutil.copy2(path, copy / path.name)
    edit(copy)
    return copy


def edit_json(path, **changes):
    record = json.loads(path.read_text())
    record.update(changes)
    path.write_text(json.dumps(record, indent=2))


cases = {
    "resize limits edited after export": lambda c: edit_json(c / "preprocessing.json", min_size=1240, max_size=1754),
    "classes swapped": lambda c: edit_json(c / "label_schema.json", id_to_code={"0": "background", "1": "price_change", "2": "exclusion"}),
    "weights hash does not match": lambda c: edit_json(c / "manifest.json", weights_sha256="0" * 64),
}
for index, (title, edit) in enumerate(cases.items()):
    try:
        load_penmark_detector(damaged_copy(f"case-{index}", edit))
        print(f"{title}: LOADED (this must not happen)")
    except ModelUnavailable as refusal:
        print(f"{title}:\n   refused -> {refusal.reason_code}: {refusal.reason_text[:150]}")
check_registry.cleanup()
del detector

## 11. Final held-out evaluation: explicit opt-in

The **test** split (74 pages) has not been read by any cell above. It exists to answer one question, once: *how does the frozen model with its frozen thresholds do on pages it has never influenced?*

Leave `FINAL_EVAL = False` while you are still changing anything: the recipe, the anchors, the thresholds, the selection rule. Looking at test results and then adjusting something turns the test split into a second validation split, and its number stops meaning what you want it to mean.

When you do run it:

- the model is the run loaded above, and the thresholds are the ones selected on validation (`selected`);
- the same `pme.evaluate_pages` function is used as for validation;
- the report is saved as `final_test_report.json` next to the figures;
- it is still **synthetic** evidence: the test pages share the layout and the handwriting fonts of the training pages. It does not replace the specification's evaluation on held-out real team-marked pages.

In [ ]:
if FINAL_EVAL:
    final_path = pmt.final_evaluation_path(RUN_EVAL_DIR, quick_run=QUICK_RUN)
    test_cache = load_m4_cache(M4_CACHE_DIR, DATASET_DIR, splits=("test",))
    test_pages = pmd.load_penmark_pages(test_cache, DATASET_DIR, splits=("test",), include_test=True)
    test_predictions = pmt.predict_pages(model, test_pages, device=device, batch_size=cfg.eval_batch_size)
    final_report = pme.evaluate_pages(test_pages, test_predictions, selected, link_config, versions=MODEL_VERSIONS,
                                      threshold_label="validation-selected")
    final_report.update(run_id=RUN_ID, thresholds_selected_on="validation", threshold_rule=THRESHOLD_RULE, provenance="synthetic",
                        evaluated_at=datetime.now(timezone.utc).isoformat())
    final_path.write_text(json.dumps(final_report, indent=1))
    print("Held-out test split (synthetic):", final_report["pages"], "| marks:", final_report["marks"])
    display(pd.DataFrame([{"class": c, **{k: v for k, v in final_report["operating_point"]["classes"][c].items()},
                           "AP@0.5": final_report["detection"][c]["ap50"], "AP@[.5:.95]": final_report["detection"][c]["ap50_95"]}
                          for c in pmd.TARGET_CLASSES]).set_index("class").round(3))
    print("Per-row detection and linking (oracle rows):")
    display(row_table(final_report["linking"]["oracle_cell_union_rows"]["detector"]["row_level"]))
    display(pd.DataFrame(final_report["operating_point"]["styles"]))
    print("Saved:", shown(final_path))
else:
    print("FINAL_EVAL is False: the test split was not read. Model and threshold selection use validation only.")

## 12. Export for serving: explicit opt-in

Set `EXPORT_FOR_SERVING = True` in the configuration cell to write the run as a registry entry at `<REGISTRY_ROOT>/penmarks/<run_id>-serving/`.

What `pmt.export_for_serving` does:

1. It **stages** the entry in a temporary folder beside the destination.
2. It loads the staged entry with the serving loader and compares its detections with the run's (the check of section 10).
3. Only then does it **rename** the folder into place. A failed check leaves nothing behind.
4. It **never overwrites**: if the destination exists, it stops.

The entry contains the files that training specification 10.2 requires:

| File | Content |
| --- | --- |
| `model.safetensors` | The weights of the best validation epoch |
| `manifest.json` | Version, `status: candidate`, hashes, seed, hardware, library versions, code revision, validation metrics, the specification targets as *not evaluated*, the M4 cache hash and M4 versions the model was trained on. A field that cannot be filled is `null` with a reason beside it |
| `label_schema.json` | `0 background, 1 exclusion, 2 price_change` |
| `preprocessing.json` | Everything the served detector applies (section 10) |
| `split_manifest.json` | Split files with hashes, per-partition counts, the grouping key |
| `evaluation_report.json` | The validation report of section 8.9 |
| `training_log.jsonl` | The per-epoch history and the effective configuration |
| `config.yaml` | The exact training configuration |
| `NOTICE.md` | Base checkpoint and licence notes, dataset provenance, limits |
| `serving_check.json` | The result of the load-and-compare check |

A **candidate** is a trained and evaluated model that nobody has accepted yet. `approved` is a decision of the lane owner and is a model-release status; it has nothing to do with approving a claim. Exporting does not change which model the application serves, and it does not edit any configuration file. The cell prints the values a person would copy.

In [ ]:
if EXPORT_FOR_SERVING:
    if QUICK_RUN:
        print("QUICK_RUN: this export checks the export plumbing. Only a validation subset was checked; do not serve this smoke-check entry.\n")
    entry = pmt.export_for_serving(RUN_DIR, EXPORT_NAME, registry_root=REGISTRY_ROOT, evaluation_report=val_report,
                                   thresholds=selected, check_pages=val_pages, created_by=CREATED_BY, device=device,
                                   threshold_rule=THRESHOLD_RULE, threshold_alternatives=threshold_alternatives,
                                     pages_used=pmt.pages_used(run_record, {"train": train_pages_all, "validation": val_pages_all}),
                                     quick_run=QUICK_RUN)
    exported = json.loads((entry / "manifest.json").read_text())
    selection = pmt.serving_selection(entry)
    print("Verified candidate:", entry)
    print("  status:", exported["status"], "| provenance:", exported["provenance"], "| weights sha256:", exported["weights_sha256"])
    print("  serving check:", json.loads((entry / "serving_check.json").read_text()))
    print("\nFor a future configs/models/penmarks.yaml (the M6 worker does not exist yet):\n")
    print(yaml.safe_dump({"detector": selection["model"]}, sort_keys=False))
    print("Rule-selected thresholds, shown alongside alternatives in section 8.2; the team chooses before any configuration change:\n")
    print(yaml.safe_dump(selection["linking"], sort_keys=False))
    print("Targets recorded in the manifest:", sorted({t["status"] for t in exported["targets"]}))
else:
    print("EXPORT_FOR_SERVING is False: nothing was exported.")
    print("The serving-path check of section 10 used a temporary entry, which has been removed.")

## 13. Interpretation, limits, glossary, next steps

### What the numbers show

- A COCO-pretrained Faster R-CNN, fine-tuned on 350 synthetic pages, **can learn to find these marks on pages like the ones it was trained on**, and to leave ticks and review notes alone. The validation tables in section 8 say how well, with counts.
- **The short anchor and resolution comparisons are inconclusive.** Default anchors and 800/1333 remain the specification defaults; the anchor comparison is smaller than the full run’s epoch-to-epoch variation.
- **Thin boxes make IoU-based scores stricter than the downstream task.** A row strike found a few pixels too tall can fail IoU 0.5 although it would be linked to the right row. The per-row numbers of section 8.7 are the ones that matter downstream.
- **The detector also proposes parts of a mark.** In the reference run, 19 of 21 false positives were parts of a found composite mark; two were poorly localised (section 8.5). The labels treat cross plus remark as one mark; whether to label and detect the parts separately is a labelling decision for the team.
- **The row geometry strongly affects linking.** The reference exclusion recall is 16/54 with full table bands, 49/54 with printed-cell unions, and 33/54 with actual M5 rows. Price-change counts are 46/46, 46/46 and 43/46. These are three distinct measurements, not evidence that the linker alone limits exclusions. Establish which boxes M5 supplies before deciding whether to tune bands.
- **Threshold 0.90 has no plateau margin for exclusions.** Validation chose the epoch, threshold and reported operating point. Compare the configured 0.50 alternative and the claim-half checks; the team chooses the recall/false-positive trade-off. No threshold configuration is changed.
- The exported entry reproduces the trained model through the serving loader (section 10).

### A recorded run

For orientation, these are the numbers of one full run of this notebook's default recipe (run `penmarks-fasterrcnn-20261012T145839Z-8529778e`, 2026-10-12, Apple M4 Max with MPS, 20 epochs, best epoch 17, 37 minutes of training while another job shared the GPU). They are **synthetic validation evidence**: 76 validation pages with 100 marks. A repeat run will differ slightly.

| Measure (validation) | exclusion | price change |
| --- | ---: | ---: |
| Marks | 54 | 46 |
| Selected score threshold (proposed) | 0.90 | 0.59 |
| Detection recall at IoU 0.5 | 53/54 = 0.981 | 46/46 = 1.000 |
| Detection precision at IoU 0.5 | 53/74 = 0.716 | 46/46 = 1.000 |
| AP@0.5 | 0.849 | 1.000 |
| AP@[.5:.95] | 0.588 | 0.896 |
| Row recall: printed-cell union (primary) | 49/54 | 46/46 |
| Row recall: full table band (contrast) | 16/54 | 46/46 |
| Row recall: actual M5 candidate | 33/54 | 43/46 |
| Oracle detections, cell union | 49/54 | 46/46 |
| Oracle detections, table band | 12/54 | 46/46 |
| Oracle detections, actual M5 | 30/54 | 43/46 |

All row precisions above are 100%. The actual M5 file contains 38 estimate pages and 192 rows; 14 of 206 true rows are unavailable on two unreadable pages. False positives: 19 parts of found marks and two poorly localised boxes; none on mark-free pages or distractors. Current proposal, conflict and unlinked counts are printed in section 8.7.

The corrected reference copy is `artifacts/evaluation/notebook-validation/M6_Document_Worker-reload-20261013.ipynb` (and HTML). Its weights were trained on 2026-10-12 by the earlier notebook revision. This correction pass uses saved weights for quick and full reloads. The training loop and optimization walkthrough were skipped; no optimization was run. The replacement candidate is `penmarks/penmarks-fasterrcnn-20261012T145839Z-8529778e-serving-r2`.

### What the numbers do not show

- **Nothing about real pages.** The ink is rendered from three handwriting fonts in a handful of pen colours, on one page layout, on flat clean scans. Real surveyors write differently, on photographed pages with shadows, folds and perspective.
- **Nothing about an unseen writer or an unseen layout.** All three fonts and the one layout are in training, validation and test.
- **Not the specification's targets.** Those are: exclusion per-row detection and linking precision and recall at least 0.90, price-change detection precision and recall at least 0.85, **on held-out real team-marked pages**. Such pages do not exist yet, so the targets are recorded as *not evaluated*. A high synthetic number is not a met target.
- **Only one M5 candidate is measured.** Its rows are real model outputs on synthetic validation, not real-page evidence.
- **Not amount reading.** A `price_change` detection is a location. The amount is typed by a person (reading it automatically is stretch goal S2, separate and optional).
- Validation is small: 100 marks, 6 to 24 per style. Differences of one or two marks are noise.
- The thresholds, the anchors, the resolution, the augmentation and the selection rule are all **proposed**, chosen on synthetic validation data.

### Glossary

| Term | Meaning |
| --- | --- |
| Anchor | A reference box at a fixed position, size and shape from which the region proposal network predicts |
| AP (average precision) | Area under the precision-recall curve; AP@0.5 uses IoU 0.5, AP@[.5:.95] averages ten IoU requirements |
| Backbone | The convolutional network (ResNet-50) that turns pixels into feature maps |
| Box head | The second stage: classifies each proposal and refines its box |
| Candidate | A trained, evaluated model that the lane owner has not accepted yet |
| Corrected page | M4's output image: the page after its geometry correction; the detector's input frame |
| Distractor | A pen mark that is not a target (tick, review note) |
| Epoch | One pass over all training pages |
| False positive / false negative | A detection with no matching mark / a mark with no matching detection |
| Fine-tuning | Continuing the training of a pretrained network on a new task |
| FPN (feature pyramid network) | Combines feature maps of several resolutions so that small and large objects are both found |
| Frozen batch norm | A normalisation layer whose statistics are fixed; its `eps` is not stored with the weights |
| IoU | Intersection over union of two boxes: 1 identical, 0 disjoint |
| Linker | The rule-based step that assigns a detected mark to a printed row |
| NMS (non-maximum suppression) | Keeps the best-scoring box and drops others that overlap it |
| Oracle rows / oracle detections | Ground-truth rows / boxes used in place of a model's output, to isolate one stage |
| Pending | The state of every proposed mark until a surveyor decides |
| Precision / recall | Share of detections that are right / share of marks that are found |
| Registry entry | The folder a worker loads a model from, with its manifest and records |
| RoI align | Cuts a fixed-size grid of features out of a feature map for one proposal |
| RPN (region proposal network) | The first stage: scores anchors and proposes regions |
| Score floor / threshold | The lowest score the detector returns / the score from which a detection is acted on |

### Next steps

1. **Real marked pages.** Collect and label team-marked physical pages, with writers, templates and pages assigned to development, validation and test *before* any tuning (training specification 8.1). This is the only route to the acceptance numbers.
2. **Stage 2 fine-tuning** on synthetic plus the real development pages, reported separately from stage 1.
3. **The M6 worker**: the Kafka consumer, the artifact store, the detection artifact and the server-rendered overlay. `load_penmark_detector` is the piece of it that exists.
4. **M5 row geometry:** inspect the measured candidate rows, including missing rows and OCR boxes that make a union taller.
5. **A team decision on row geometry first**, then whether its linking band needs tuning, plus the threshold trade-off and composite-mark labelling policy.
6. **Richer synthetic pages** if they are still needed: more layouts, more writers, photographic distortion upstream of M4.

Run outputs: checkpoints and history in `artifacts/models/penmarks/<run_id>/`, figures and reports in `artifacts/evaluation/<run_id>/`, an exported candidate in `artifacts/models/penmarks/<run_id>-serving/`. All three are ignored by Git.